# Scutellum Height Oscillation and Walking vs. Running

Companion to `Joint_Kinematics_Analysis.ipynb`. Tests whether *Drosophila* locomotion follows an inverted-pendulum (walking) or spring-mass (running) model by measuring the phase relationship between thorax height and forward-speed oscillations across the full speed range.


In [ ]:
### Setup — paths, imports
import sys
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import h5py
from scipy import signal
from scipy.ndimage import uniform_filter1d, gaussian_filter1d
from scipy.signal import butter, filtfilt, hilbert
from scipy.stats import spearmanr, pearsonr, circmean, circstd
import warnings
warnings.filterwarnings('ignore', category=RuntimeWarning)

# Add 3d_tracking_dataset repo root so utils can be imported
_nb_dir = Path().resolve()
_repo_root = _nb_dir.parent if _nb_dir.name == 'notebooks' else _nb_dir
if str(_repo_root) not in sys.path:
    sys.path.insert(0, str(_repo_root))
import utils.io_dict_to_hdf5 as ioh5

# ── Edit these ────────────────────────────────────────────────────────────────

# H5_PATH    = Path("/home/user/3D_tracking_paper/IK_outputs/running_videos/free_running/Data_analysis/analysis/ik_output_combined_v1_free_running.h5")
H5_PATH = Path('/home/user/3D_tracking_paper/IK_outputs/running_videos/free_running/Data_analysis/analysis/v1/ik_output_combined_v1_free_running.h5') # all flies
OUTPUT_DIR = Path('/home/user/3D_tracking_paper/output/free_running')
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

FPS           = 800          # Hz
LEGS          = ['T1_left', 'T1_right', 'T2_left', 'T2_right', 'T3_left', 'T3_right']
REFERENCE_LEG = 'T1_left'
HEADING_BODIES = ['thorax']
SCALE         = 10.0         # model units → mm
SMOOTH_SIGMA  = 6            # Gaussian sigma (frames) for Hilbert phase


In [ ]:
### Paper figure defaults — Arial 6 pt, TrueType fonts
# This cell sets global rcParams for all paper figures in the notebook.
# Re-run if you restart the kernel.
import matplotlib as _mpl
import matplotlib.font_manager as _fm
_fm.fontManager.addfont("/usr/share/fonts/truetype/msttcorefonts/Arial.ttf")
_fm.fontManager.addfont("/usr/share/fonts/truetype/msttcorefonts/Arial_Bold.ttf")
_mpl.rcParams.update({
    'font.family':      'Arial',
    'font.size':        6,
    'axes.labelsize':   6,
    'axes.titlesize':   6,
    'xtick.labelsize':  6,
    'ytick.labelsize':  6,
    'legend.fontsize':  6,
    'pdf.fonttype':     42,
    'ps.fonttype':      42,
    'axes.linewidth':   0.5,
    'xtick.major.width': 0.5,
    'ytick.major.width': 0.5,
    'xtick.major.size':  2,
    'ytick.major.size':  2,
    'lines.linewidth':  0.75,
})
print("Paper rcParams set: Arial 6 pt, pdf.fonttype=42")


## Build df_valid from HDF5

Builds the per-frame DataFrame from the STAC IK output HDF5. Set `H5_PATH` in the Setup cell above, then run the next cell.

**Columns produced:**

| Column | Description |
|---|---|
| `bout_id` | HDF5 group key |
| `fly_id` | first token of `bout_id` split on `_` (edit `_parse_fly_id` if needed) |
| `frame` | 0-based frame index within bout |
| `thorax_z` | `xpos[:,1,2]` model units (Cell H1 multiplies by 10 → mm) |
| `forward_speed` | thorax XY displacement × FPS (mm/s) |
| `T1_left_phase` | linear phase in [−π, π] per T1L stride cycle; NaN outside complete cycles |
| `n_legs_stance` | legs in stance (0–6) per frame |
| `step_cycle_id` | integer cycle index from T1L swing onsets |
| `step_cycle_mean_speed` | mean forward_speed within each complete stride cycle |

**Site indices** (fruitfly_v1, verified from `fruitfly_v1_free.xml`):
`T1L=18, T1R=25, T2L=31, T2R=37, T3L=43, T3R=49` in `xpos_egocentric[:,idx,:]`

In [ ]:
### Build df_valid — pipeline functions (mirrors Joint_Kinematics_Analysis)

# ── Data loading ──────────────────────────────────────────────────────────────
def load_ik_bouts(h5_path):
    d = ioh5.load(h5_path)
    bout_keys    = sorted(k for k in d if k.startswith('bout_'))
    fly_ids      = np.array(d['info']['fly_ids'])
    clip_lengths = np.array(d['info']['clip_lengths'])
    print(f"Loaded {len(bout_keys)} bouts | {len(np.unique(fly_ids))} unique flies")
    print(f"Frames/bout: min={clip_lengths.min()}, max={clip_lengths.max()}, "
          f"total={clip_lengths.sum():,}")
    return d, bout_keys, fly_ids, clip_lengths

_LEG_SHORT = {
    'T1_left':  'T1L', 'T1_right': 'T1R',
    'T2_left':  'T2L', 'T2_right': 'T2R',
    'T3_left':  'T3L', 'T3_right': 'T3R',
}

def find_leg_tip_site_indices(site_names, legs):
    indices = {}
    for leg in legs:
        short = _LEG_SHORT.get(leg, leg)
        patterns = [f"{short}_TaTip", f"{leg}_TaTip", f"claw_{leg}", f"{leg}_claw"]
        match = None
        for p in patterns:
            match = next((i for i, n in enumerate(site_names) if p in n), None)
            if match is not None:
                break
        indices[leg] = match
        if match is None:
            print(f"Warning: no tip site for {leg}")
    return indices

def bouts_to_dataframe(bout_dict, bout_keys, fly_ids, joint_list,
                       names_xpos, heading_bodies,
                       egocentric_site_names, leg_tip_site_indices):
    frames = []
    for bout_idx, (key, fly_id) in enumerate(zip(bout_keys, fly_ids)):
        b    = bout_dict[key]
        xpos = np.array(b['xpos'])
        T    = xpos.shape[0]
        row  = {
            'frame':    np.arange(T),
            'bout_idx': np.full(T, bout_idx, dtype=np.int32),
            'bout_id':  np.full(T, key, dtype=object),
            'fly_id':   np.full(T, fly_id, dtype=object),
        }
        if 'qpos' in b:
            qpos = np.array(b['qpos'])
            for leg, joint, idx in joint_list:
                row[f"{leg}_{joint}"] = qpos[:, idx]
        for body in heading_bodies:
            if body in names_xpos:
                bidx = names_xpos.index(body)
                row[f"{body}_x"] = xpos[:, bidx, 0]
                row[f"{body}_y"] = xpos[:, bidx, 1]
                row[f"{body}_z"] = xpos[:, bidx, 2]
        if 'xpos_egocentric' in b:
            xpos_ego = np.array(b['xpos_egocentric'])
            for leg, site_idx in leg_tip_site_indices.items():
                if site_idx is not None and site_idx < xpos_ego.shape[1]:
                    row[f"{leg}_tip_x_ego"] = xpos_ego[:, site_idx, 0]
                    row[f"{leg}_tip_y_ego"] = xpos_ego[:, site_idx, 1]
                    row[f"{leg}_tip_z_ego"] = xpos_ego[:, site_idx, 2]
        _CLAW_NAMES = {
            'T1_left': 'claw_T1_left', 'T1_right': 'claw_T1_right',
            'T2_left': 'claw_T2_left', 'T2_right': 'claw_T2_right',
            'T3_left': 'claw_T3_left', 'T3_right': 'claw_T3_right',
        }
        for leg, cname in _CLAW_NAMES.items():
            if cname in names_xpos:
                ci = names_xpos.index(cname)
                row[f"{leg}_tip_x_world"] = xpos[:, ci, 0]
                row[f"{leg}_tip_y_world"] = xpos[:, ci, 1]
                row[f"{leg}_tip_z_world"] = xpos[:, ci, 2]
        frames.append(pd.DataFrame(row))
    df = pd.concat(frames, ignore_index=True)
    print(f"Built DataFrame: {len(df):,} frames, {len(df.columns)} columns")
    return df

# ── Preprocessing ─────────────────────────────────────────────────────────────
def _gaussian_smooth(arr, sigma=4):
    mask = np.isnan(arr)
    out  = gaussian_filter1d(np.where(mask, 0., arr), sigma)
    cnt  = gaussian_filter1d((~mask).astype(float), sigma)
    out[cnt > 0] /= cnt[cnt > 0]
    out[mask]     = np.nan
    return out

def swing_onsets_from_stance(series):
    ss = series.values.astype(int)
    return np.where(np.diff(ss, prepend=ss[0]) == -1)[0]

def compute_swing_stance_tip_speed(df, legs, fps=800, thresh_frac=0.20):
    df = df.copy()
    for leg in legs:
        xcol, ycol, zcol = f"{leg}_tip_x_world", f"{leg}_tip_y_world", f"{leg}_tip_z_world"
        if zcol not in df.columns:
            continue
        ss = np.full(len(df), np.nan)
        for _bid, grp in df.groupby('bout_id', sort=False):
            idx = grp.index
            n   = len(grp)
            if n < 5:
                continue
            def _vel(col):
                v = grp[col].values.astype(float) if col in grp.columns else np.zeros(n)
                return np.gradient(v) * fps * 10
            spd = np.sqrt(_vel(xcol)**2 + _vel(ycol)**2 + _vel(zcol)**2)
            spd = _gaussian_smooth(spd, sigma=4)
            thr = thresh_frac * np.nanmax(spd)
            ss[idx] = (spd <= thr).astype(float)
        valid = ~np.isnan(ss)
        df.loc[valid, f"{leg}_swing_stance"] = ss[valid].astype(int)
    return df

def compute_n_legs_stance(df, legs):
    df = df.copy()
    cols = [f"{leg}_swing_stance" for leg in legs if f"{leg}_swing_stance" in df.columns]
    if cols:
        df['n_legs_stance'] = df[cols].sum(axis=1)
    return df

# ── Hilbert phase ─────────────────────────────────────────────────────────────
def compute_hilbert_phase_bout(positions_xyz, fps, smooth_sigma,
                               mask_stance=False, stance_speed_frac=0.20):
    """Per-frame Hilbert phase from 3-D tip speed.

    phase=0: mid-swing (peak speed).  phase=±π: mid-stance.
    Speed is mean-centered before Hilbert so phase spans full [-π, π].
    """
    T = len(positions_xyz)
    if T < 10:
        return np.full(T, np.nan), np.zeros(T)
    xyz = positions_xyz.astype(float).copy()
    for dim in range(3):
        col = xyz[:, dim]
        nans = np.isnan(col)
        if nans.any() and (~nans).sum() > 1:
            col[nans] = np.interp(np.flatnonzero(nans), np.flatnonzero(~nans), col[~nans])
        xyz[:, dim] = col
    vel = np.gradient(xyz, axis=0) * fps
    spd = gaussian_filter1d(np.linalg.norm(vel, axis=1), sigma=smooth_sigma)
    ph  = np.angle(hilbert(spd - np.nanmean(spd)))
    ph[:2] = np.nan; ph[-2:] = np.nan
    if mask_stance:
        ph[spd <= stance_speed_frac * np.nanmax(spd)] = np.nan
    return ph, spd

def swing_onsets_from_hilbert(phase):
    """Detect step-cycle boundaries (mid-swing) from 2π increments in unwrapped Hilbert phase."""
    finite_mask = np.isfinite(phase)
    if finite_mask.sum() < 4:
        return np.array([], dtype=int)
    finite_idx = np.where(finite_mask)[0]
    unwrapped  = np.unwrap(phase[finite_mask])
    jumps = np.where(np.diff(np.floor(unwrapped / (2 * np.pi))) > 0)[0]
    return finite_idx[jumps]

def compute_hilbert_phases(df, legs, fps, smooth_sigma):
    """Compute per-frame Hilbert phase for each leg using world-frame 3-D tip speed.

    Replaces linear-ramp compute_phases_from_swing.
    Convention: phase=0 mid-swing, phase=±π mid-stance.
    Processed per bout to avoid cross-bout edge effects.
    """
    df = df.copy()
    for leg in legs:
        df[f'{leg}_phase'] = np.nan
        x_col = f'{leg}_tip_x_world'
        y_col = f'{leg}_tip_y_world'
        z_col = f'{leg}_tip_z_world'
        if z_col not in df.columns:
            continue
        for bout_id, grp in df.groupby('bout_id', sort=False):
            xyz = grp[[x_col, y_col, z_col]].values
            ph, _ = compute_hilbert_phase_bout(xyz, fps, smooth_sigma, mask_stance=False)
            df.loc[grp.index, f'{leg}_phase'] = ph
    return df


def liftoff_from_hilbert(phase):
    """Upward crossing of phase = -π/2 → liftoff (swing onset)."""
    finite_mask = np.isfinite(phase)
    if finite_mask.sum() < 4:
        return np.array([], dtype=int)
    finite_idx = np.where(finite_mask)[0]
    unwrapped  = np.unwrap(phase[finite_mask])
    increments = np.diff(np.floor((unwrapped + np.pi / 2) / (2 * np.pi)))
    return finite_idx[np.where(increments > 0)[0]]


def landing_from_hilbert(phase):
    """Upward crossing of phase = +π/2 → landing (stance onset)."""
    finite_mask = np.isfinite(phase)
    if finite_mask.sum() < 4:
        return np.array([], dtype=int)
    finite_idx = np.where(finite_mask)[0]
    unwrapped  = np.unwrap(phase[finite_mask])
    increments = np.diff(np.floor((unwrapped - np.pi / 2) / (2 * np.pi)))
    return finite_idx[np.where(increments > 0)[0]]


def compute_swing_stance_from_hilbert(df, legs):
    """Derive binary swing/stance from Hilbert phase.

    swing  (0): |phase| < π/2  — near mid-swing (peak speed)
    stance (1): |phase| >= π/2 — near mid-stance (speed trough)

    Replaces compute_swing_stance_tip_speed. Must be called after
    compute_hilbert_phases.
    """
    df = df.copy()
    for leg in legs:
        phase_col = f'{leg}_phase'
        if phase_col not in df.columns:
            continue
        ph = df[phase_col].values
        ss = np.where(np.isfinite(ph), (np.abs(ph) >= np.pi / 2).astype(float), np.nan)
        df[f'{leg}_swing_stance'] = ss
    return df



def compute_heading_and_velocity(df, fps=800, body='thorax', smooth_sigma=2):
    df = df.copy()
    x_col, y_col = f"{body}_x", f"{body}_y"
    if x_col not in df.columns or y_col not in df.columns:
        print(f"Warning: position columns not found for {body}")
        return df
    heading_arr   = np.full(len(df), np.nan)
    speed_arr     = np.full(len(df), np.nan)
    turn_rate_arr = np.full(len(df), np.nan)
    for bout_id, grp in df.groupby('bout_id'):
        idx = grp.index
        x   = gaussian_filter1d(grp[x_col].values.astype(float), sigma=smooth_sigma)
        y   = gaussian_filter1d(grp[y_col].values.astype(float), sigma=smooth_sigma)
        dx  = np.gradient(x) * fps
        dy  = np.gradient(y) * fps
        heading   = np.arctan2(dy, dx)
        heading_u = np.unwrap(heading)
        heading_s = gaussian_filter1d(heading_u, sigma=smooth_sigma)
        heading_s = (heading_s + np.pi) % (2 * np.pi) - np.pi
        heading_arr[idx]   = heading_s
        speed_arr[idx]     = np.sqrt(dx**2 + dy**2) * 10   # mm/s
        turn_rate_arr[idx] = np.gradient(heading_s) * fps
    df['heading']       = heading_arr
    df['forward_speed'] = speed_arr
    df['turning_rate']  = turn_rate_arr
    return df

def compute_step_cycle_speed(df, fps=800, ref_leg='T1_left'):
    df    = df.copy()
    phase_col = f"{ref_leg}_phase"
    spd_col = 'forward_speed'
    sc_id  = np.full(len(df), np.nan)
    sc_spd = np.full(len(df), np.nan)
    if phase_col not in df.columns:
        df['step_cycle_id']         = sc_id
        df['step_cycle_mean_speed'] = sc_spd
        return df
    for bout_id, grp in df.groupby('bout_id', sort=False):
        idx    = grp.index
        onsets = liftoff_from_hilbert(grp[phase_col].values)
        if len(onsets) < 2:
            continue
        spd = grp[spd_col].values.astype(float) if spd_col in grp.columns \
              else np.full(len(grp), np.nan)
        for k in range(len(onsets) - 1):
            t0, t1 = onsets[k], onsets[k + 1]
            sc_id[idx[t0:t1]]  = k
            sc_spd[idx[t0:t1]] = np.nanmean(spd[t0:t1])
    df['step_cycle_id']         = sc_id
    df['step_cycle_mean_speed'] = sc_spd
    return df

# ── Build df_valid ────────────────────────────────────────────────────────────
bout_dict, bout_keys, fly_ids, clip_lengths = load_ik_bouts(H5_PATH)
names_qpos            = list(bout_dict['info']['names_qpos'])
names_xpos            = list(bout_dict['info']['names_xpos'])
egocentric_site_names = list(bout_dict['info']['site_names_egocentric'])

leg_tip_site_indices = find_leg_tip_site_indices(egocentric_site_names, LEGS)
print(f"Leg tip site indices: {leg_tip_site_indices}")

# No joint angles needed for this analysis — pass empty joint_list
df = bouts_to_dataframe(
    bout_dict, bout_keys, fly_ids, joint_list=[],
    names_xpos=names_xpos,
    heading_bodies=HEADING_BODIES,
    egocentric_site_names=egocentric_site_names,
    leg_tip_site_indices=leg_tip_site_indices,
)

df = compute_hilbert_phases(df, LEGS, fps=FPS, smooth_sigma=SMOOTH_SIGMA)
df = compute_swing_stance_from_hilbert(df, LEGS)
df = compute_n_legs_stance(df, LEGS)
df = compute_heading_and_velocity(df, fps=FPS, body=HEADING_BODIES[0])
df = compute_step_cycle_speed(df, fps=FPS, ref_leg=REFERENCE_LEG)

# thorax_z is in model metres; * SCALE gives mm  (SCALE=10 → mm)
df_valid = df.copy()
print(f"\ndf_valid: {len(df_valid):,} rows | "
      f"{df_valid['bout_id'].nunique()} bouts | "
      f"{df_valid['fly_id'].nunique()} flies")
print(f"  thorax_z:              mean={df_valid['thorax_z'].mean():.4f} model-m  "
      f"({df_valid['thorax_z'].mean()*SCALE:.3f} mm)")
print(f"  forward_speed (mm/s):  {df_valid['forward_speed'].mean():.1f} ± "
      f"{df_valid['forward_speed'].std():.1f}")
print(f"  T1_left_phase valid:   "
      f"{df_valid['T1_left_phase'].notna().mean()*100:.1f}% of frames")
print(f"  step_cycle_mean_speed: "
      f"{df_valid['step_cycle_mean_speed'].notna().mean()*100:.1f}% of frames")


In [ ]:
### Cell H1 — Add detrended thorax height to df_valid

df_valid['thorax_z_mm'] = df_valid['thorax_z'] * 10.0   # model-m → mm

BASELINE_FRAMES = int(0.100 * FPS)   # 400 frames — 500 ms rolling window

df_valid['thorax_z_baseline']  = np.nan
df_valid['thorax_z_detrended'] = np.nan

for bid, grp in df_valid.groupby('bout_id', sort=False):
    idx      = grp.index
    z_mm     = grp['thorax_z_mm'].values
    baseline = uniform_filter1d(z_mm, size=BASELINE_FRAMES, mode='nearest')
    df_valid.loc[idx, 'thorax_z_baseline']  = baseline
    df_valid.loc[idx, 'thorax_z_detrended'] = z_mm - baseline

# Bandpass oscillatory component (5–50 Hz — stride frequency band at 800 Hz)
b_bp, a_bp = butter(3, [5 / (FPS / 2), 50 / (FPS / 2)], btype='band')

df_valid['thorax_z_osc'] = np.nan
for bid, grp in df_valid.groupby('bout_id', sort=False):
    idx = grp.index
    if len(idx) < 30:
        continue
    df_valid.loc[idx, 'thorax_z_osc'] = filtfilt(b_bp, a_bp, grp['thorax_z_mm'].values)

print(f"thorax_z_mm:        mean={df_valid['thorax_z_mm'].mean():.3f} mm  "
      f"std={df_valid['thorax_z_mm'].std():.3f} mm")
print(f"thorax_z_detrended: mean≈0  std={df_valid['thorax_z_detrended'].std():.4f} mm")
print(f"thorax_z_osc:       mean≈0  std={df_valid['thorax_z_osc'].std():.4f} mm")

In [ ]:
# ── Valid running cycles per fly ──────────────────────────────────────────────
n_bouts = df_valid.groupby('fly_id')['bout_idx'].nunique().rename('n_bouts')

# Count complete step cycles: unique (fly_id, bout_idx, step_cycle_id) where cycle_id is valid
cycles = df_valid.dropna(subset=['step_cycle_id'])
n_cycles = (cycles.groupby('fly_id')
                  .apply(lambda g: g.groupby(['bout_idx', 'step_cycle_id']).ngroups,
                         include_groups=False)
                  .rename('n_cycles'))

summary = pd.concat([n_bouts, n_cycles], axis=1)
summary['cycles_per_bout'] = (summary['n_cycles'] / summary['n_bouts']).round(1)
print(summary.to_string())

In [ ]:
### Cell H2 — Phase-averaged height oscillation

N_BINS      = 48
N_QUANTILES = 6   # ← change this: 3=terciles, 4=quartiles, 5=quintiles, etc.

phase_bins  = np.linspace(-np.pi, np.pi, N_BINS + 1)
bin_centers = 0.5 * (phase_bins[:-1] + phase_bins[1:])
bin_idx     = np.clip(np.digitize(df_valid['T1_left_phase'], phase_bins) - 1, 0, N_BINS - 1)

# Dynamic labels
_q_labels = [f'Q{i+1}' for i in range(N_QUANTILES)]
if N_QUANTILES == 3:
    _q_labels = ['slow', 'medium', 'fast']
elif N_QUANTILES == 4:
    _q_labels = ['slow', 'med-slow', 'med-fast', 'fast']

speed_quantiles  = pd.qcut(df_valid['step_cycle_mean_speed'], q=N_QUANTILES,
                            labels=_q_labels)
df_valid['_sq'] = speed_quantiles

# Mean speed per quantile (used later for peak analysis)
_mean_speed_per_q = (df_valid.groupby('_sq', observed=True)['step_cycle_mean_speed']
                     .mean().to_dict())

# Compute mean touchdown phase (swing→stance transition) from T1_left data
_td_phases_list = []
for _bid, _grp in df_valid.groupby('bout_id', sort=False):
    if 'T1_left_swing_stance' not in _grp.columns:
        continue
    _ss = _grp['T1_left_swing_stance'].values.astype(int)
    _ph = _grp['T1_left_phase'].values
    _td_frames = np.where(np.diff(_ss, prepend=_ss[0]) == 1)[0]
    for _tf in _td_frames:
        if _tf < len(_ph) and not np.isnan(_ph[_tf]):
            _td_phases_list.append(_ph[_tf])

mean_td_phase = circmean(_td_phases_list, high=np.pi, low=-np.pi) if _td_phases_list else None
print(f"Mean touchdown phase: {mean_td_phase:.3f} rad  ({np.degrees(mean_td_phase):.1f}°)" if mean_td_phase is not None else "No touchdown phase computed")

z_vals = df_valid['thorax_z_detrended'].values * 1000   # mm → µm

# ── Bootstrap helper (resamples step cycles) ─────────────────────────────────
def _bootstrap_tercile_ci(tercile_mask, n_boot=1000, ci=95, seed=42):
    """Bootstrap CI over step cycles for a subset of df_valid rows.

    Resamples STEP CYCLES (not frames) to respect within-cycle correlations.
    Returns (mean, lo, hi) each shape (N_BINS,).
    """
    sub = df_valid[tercile_mask].dropna(subset=['step_cycle_id', 'T1_left_phase'])
    cycle_groups = list(sub.groupby(['bout_id', 'step_cycle_id']))
    n_cyc = len(cycle_groups)
    if n_cyc < 2:
        nans = np.full(N_BINS, np.nan)
        return nans, nans, nans

    cyc_mat = np.full((n_cyc, N_BINS), np.nan)
    for ci_i, (_, grp) in enumerate(cycle_groups):
        grp_pos = df_valid.index.get_indexer(grp.index)
        _b = bin_idx[grp_pos]
        _z = grp['thorax_z_detrended'].values * 1000
        for b in range(N_BINS):
            vals = _z[_b == b]
            if len(vals):
                cyc_mat[ci_i, b] = np.nanmean(vals)

    rng  = np.random.default_rng(seed)
    boot = np.full((n_boot, N_BINS), np.nan)
    for i in range(n_boot):
        idx     = rng.integers(0, n_cyc, size=n_cyc)
        boot[i] = np.nanmean(cyc_mat[idx], axis=0)

    half = (100 - ci) / 2
    mean = np.nanmean(cyc_mat, axis=0)
    lo   = np.nanpercentile(boot, half,       axis=0)
    hi   = np.nanpercentile(boot, 100 - half, axis=0)
    return mean, lo, hi

colors_q = plt.cm.viridis(np.linspace(0.2, 0.9, N_QUANTILES))

fig, axes = plt.subplots(1, 2, figsize=(16, 5))

# ── Left: overall phase average ───────────────────────────────────────────────
z_phase = np.array([z_vals[bin_idx == b].mean() for b in range(N_BINS)])
z_std   = np.array([z_vals[bin_idx == b].std()  for b in range(N_BINS)])
axes[0].fill_between(bin_centers, z_phase - z_std, z_phase + z_std, alpha=0.3, label='±STD')
axes[0].plot(bin_centers, z_phase, lw=2)
axes[0].axhline(0, color='k', lw=0.5, ls='--')
axes[0].axvline(-np.pi/2, color='blue', ls=':', lw=1, label='T1L liftoff (-π/2)')
axes[0].axvline(-np.pi,  color='gray',  ls=':', lw=0.8, alpha=0.5, label='mid-stance (±π)')
if mean_td_phase is not None:
    axes[0].axvline(mean_td_phase, color='orange', ls='--', lw=1,
                    label=f'T1L touchdown (mean ≈ {mean_td_phase:.2f} rad)')
axes[0].set_xlabel('T1_left phase (rad)'); axes[0].set_ylabel('Detrended thorax Z (µm)')
axes[0].set_title('Phase-averaged body height (all speeds)')
axes[0].set_xticks([-np.pi, -np.pi/2, 0, np.pi/2, np.pi])
axes[0].set_xticklabels(['-π', '-π/2', '0', 'π/2', 'π'])
axes[0].legend(fontsize=8)

# ── Right: by speed quantile + bootstrap 95% CI ───────────────────────────────
_z_phase_q_stored = {}
for qi, (qlabel, _) in enumerate(df_valid.groupby('_sq', observed=True)):
    mask = df_valid['_sq'] == qlabel
    qmean, qlo, qhi = _bootstrap_tercile_ci(mask)
    n_cyc = len(list(df_valid[mask].dropna(subset=['step_cycle_id'])
                     .groupby(['bout_id', 'step_cycle_id'])))
    axes[1].fill_between(bin_centers, qlo, qhi,
                         color=colors_q[qi], alpha=0.25)
    axes[1].plot(bin_centers, qmean,
                 color=colors_q[qi], lw=2,
                 label=f'{qlabel}  (n={n_cyc} cycles)')
    _z_phase_q_stored[qlabel] = qmean

axes[1].axhline(0, color='k', lw=0.5, ls='--')
axes[1].axvline(-np.pi/2, color='blue', ls=':', lw=1, label='T1L liftoff (-π/2)')
axes[1].axvline(-np.pi,  color='gray',  ls=':', lw=0.8, alpha=0.5)
if mean_td_phase is not None:
    axes[1].axvline(mean_td_phase, color='orange', ls='--', lw=1)
axes[1].set_xlabel('T1_left phase (rad)'); axes[1].set_ylabel('Detrended thorax Z (µm)')
axes[1].set_title(f'Phase-averaged body height by speed quantile (N={N_QUANTILES})\n'
                  '(shading = 95% bootstrap CI, resampled over step cycles)')
axes[1].set_xticks([-np.pi, -np.pi/2, 0, np.pi/2, np.pi])
axes[1].set_xticklabels(['-π', '-π/2', '0', 'π/2', 'π'])
axes[1].legend(fontsize=8)

plt.suptitle('Body height oscillation vs. stride phase\n'
             '(phase: ±π = mid-stance  |  -π/2 = liftoff  |  0 = mid-swing  |  +π/2 = touchdown\n'
             'phase advances slowly during stance → equal phase width ≠ equal time)',
             fontsize=11, fontweight='bold')
plt.tight_layout()
plt.savefig(OUTPUT_DIR / 'height_phase_avg.pdf', bbox_inches='tight')
plt.show()

_z_phase_stored = z_phase.copy()

In [ ]:
### Cell H2c — Peak phase shift with speed

# Requires H2 to have run (provides _z_phase_q_stored, bin_centers, _mean_speed_per_q,
#  N_BINS, N_QUANTILES, colors_q, mean_td_phase, OUTPUT_DIR)

from scipy.signal import find_peaks
from scipy.stats import spearmanr, linregress

# ── Per-quantile mean touchdown phase ─────────────────────────────────────────
# Touchdown = swing→stance transition (T1_left_swing_stance 0→1)
_td_phase_per_q = {}
for qi, qlabel in enumerate(df_valid.groupby('_sq', observed=True).groups.keys()):
    _td_list = []
    q_mask = df_valid['_sq'] == qlabel
    for _bid, _grp in df_valid[q_mask].groupby('bout_id', sort=False):
        if 'T1_left_swing_stance' not in _grp.columns:
            continue
        _ss = _grp['T1_left_swing_stance'].values.astype(int)
        _ph = _grp['T1_left_phase'].values
        _td_frames = np.where(np.diff(_ss, prepend=_ss[0]) == 1)[0]
        for _tf in _td_frames:
            if _tf < len(_ph) and not np.isnan(_ph[_tf]):
                _td_list.append(_ph[_tf])
    if _td_list:
        _td_phase_per_q[qlabel] = circmean(_td_list, high=np.pi, low=-np.pi)
        print(f"  [{qlabel}] mean touchdown phase: "
              f"{_td_phase_per_q[qlabel]:.3f} rad  "
              f"({np.degrees(_td_phase_per_q[qlabel]):.1f}°)  "
              f"(n={len(_td_list)} touchdowns)")

# ── Find top-2 peaks in each quantile's mean phase curve ─────────────────────
PEAK_SMOOTH_SIGMA    = 1.5   # Gaussian smoothing before peak detection (bins)
PEAK_MIN_DISTANCE    = 8     # minimum bins between peaks
PEAK_MIN_PROMINENCE  = 0.5   # µm — ignore tiny wiggles below this

peak_records = []
_smoothed_q  = {}

quantile_order = list(df_valid.groupby('_sq', observed=True).groups.keys())

for qi, qlabel in enumerate(quantile_order):
    curve = _z_phase_q_stored.get(qlabel)
    if curve is None or np.all(np.isnan(curve)):
        continue

    # Wrap-around smoothing so peaks near ±π aren't clipped at boundary
    pad = N_BINS // 4
    padded = np.concatenate([curve[-pad:], curve, curve[:pad]])
    smoothed_padded = gaussian_filter1d(np.nan_to_num(padded, nan=0.0), PEAK_SMOOTH_SIGMA)
    smoothed = smoothed_padded[pad: pad + N_BINS]
    _smoothed_q[qlabel] = smoothed

    curve_range = np.nanmax(smoothed) - np.nanmin(smoothed)
    prominence_thr = max(PEAK_MIN_PROMINENCE, 0.10 * curve_range)

    peaks, props = find_peaks(
        smoothed,
        distance=PEAK_MIN_DISTANCE,
        prominence=prominence_thr,
    )

    if len(peaks) == 0:
        print(f"[{qlabel}] No peaks found — skipping")
        continue

    prom_order = np.argsort(props['prominences'])[::-1]
    top_peaks  = sorted(peaks[prom_order[:2]])

    mean_spd = _mean_speed_per_q.get(qlabel, np.nan)

    for pk_rank, pk_bin in enumerate(top_peaks):
        peak_records.append(dict(
            quantile       = qlabel,
            qi             = qi,
            peak_rank      = pk_rank,
            peak_bin       = pk_bin,
            peak_phase     = bin_centers[pk_bin],
            peak_phase_deg = np.degrees(bin_centers[pk_bin]),
            peak_height    = smoothed[pk_bin],
            mean_speed     = mean_spd,
        ))

df_peaks = pd.DataFrame(peak_records)
print()
print(df_peaks.to_string(index=False))

# ── Statistical test: Spearman rank correlation speed → peak phase ─────────────
print("\n── Spearman correlation: mean speed vs. peak phase ──")
for rank in [0, 1]:
    sub = df_peaks[df_peaks['peak_rank'] == rank].dropna(subset=['mean_speed', 'peak_phase'])
    if len(sub) < 3:
        print(f"  Peak {'early' if rank==0 else 'late'}: too few points (n={len(sub)})")
        continue
    r, p = spearmanr(sub['mean_speed'], sub['peak_phase'])
    slope, intercept, *_ = linregress(sub['mean_speed'], np.degrees(sub['peak_phase']))
    lbl = 'early' if rank == 0 else 'late'
    print(f"  Peak {lbl}: Spearman r={r:.3f}, p={p:.4f}  |  "
          f"OLS slope={slope:.2f} deg/(mm/s)  (n={len(sub)} quantiles)")

# ── Figure ────────────────────────────────────────────────────────────────────
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# Left: smoothed curves with detected peaks + per-quantile touchdown phases
for qi, qlabel in enumerate(quantile_order):
    if qlabel not in _smoothed_q:
        continue
    smoothed = _smoothed_q[qlabel]
    color = colors_q[qi]

    axes[0].plot(bin_centers, smoothed, color=color, lw=2, label=qlabel)

    # Per-quantile touchdown phase (dashed vertical line, same color)
    if qlabel in _td_phase_per_q:
        axes[0].axvline(_td_phase_per_q[qlabel], color=color,
                        ls='--', lw=1.2, alpha=0.8)

    # Peak markers
    pk_sub = df_peaks[df_peaks['quantile'] == qlabel]
    for _, row in pk_sub.iterrows():
        marker = '^' if row['peak_rank'] == 0 else 'v'
        axes[0].plot(row['peak_phase'], row['peak_height'],
                     marker=marker, ms=10, color=color,
                     markeredgecolor='k', markeredgewidth=0.8, zorder=5)

# Liftoff reference (−π/2, global)
axes[0].axvline(-np.pi/2, color='blue', ls=':', lw=1, alpha=0.5, label='T1L liftoff (−π/2)')
axes[0].axhline(0, color='k', lw=0.5, ls='--')
axes[0].set_xlabel('T1_left phase (rad)')
axes[0].set_ylabel('Detrended thorax Z (µm)')
axes[0].set_title('Smoothed curves + per-quantile touchdown\n'
                  '(▲=early peak, ▼=late peak, dashed=touchdown)')
axes[0].set_xticks([-np.pi, -np.pi/2, 0, np.pi/2, np.pi])
axes[0].set_xticklabels(['-π', '-π/2', '0', 'π/2', 'π'])
axes[0].legend(fontsize=8)

# Middle: peak phase vs. mean speed
colors_pk = ['steelblue', 'tomato']
labels_pk = ['Early peak', 'Late peak']
for rank in [0, 1]:
    sub = df_peaks[df_peaks['peak_rank'] == rank].sort_values('mean_speed')
    if sub.empty:
        continue
    axes[1].plot(sub['mean_speed'], np.degrees(sub['peak_phase']),
                 'o-', color=colors_pk[rank], lw=2, ms=8, label=labels_pk[rank])
    for _, row in sub.iterrows():
        axes[1].annotate(row['quantile'],
                         (row['mean_speed'], np.degrees(row['peak_phase'])),
                         textcoords='offset points', xytext=(4, 4), fontsize=7)

# Also plot touchdown phase vs. speed
_td_spd   = [_mean_speed_per_q[q] for q in quantile_order if q in _td_phase_per_q]
_td_ph_deg = [np.degrees(_td_phase_per_q[q]) for q in quantile_order if q in _td_phase_per_q]
if _td_spd:
    axes[1].plot(_td_spd, _td_ph_deg, 'D--', color='gray', lw=1.5, ms=7,
                 label='Touchdown phase', alpha=0.8)

axes[1].set_xlabel('Mean speed of quantile (mm/s)')
axes[1].set_ylabel('Phase (°)')
axes[1].set_title('Peak & touchdown phase vs. speed\n(do they shift?)')
axes[1].legend(fontsize=8)
axes[1].axhline(0, color='k', lw=0.5, ls='--', alpha=0.4)

# Right: peak amplitude vs. mean speed
for rank in [0, 1]:
    sub = df_peaks[df_peaks['peak_rank'] == rank].sort_values('mean_speed')
    if sub.empty:
        continue
    axes[2].plot(sub['mean_speed'], sub['peak_height'],
                 's-', color=colors_pk[rank], lw=2, ms=8, label=labels_pk[rank])

axes[2].axhline(0, color='k', lw=0.5, ls='--', alpha=0.4)
axes[2].set_xlabel('Mean speed of quantile (mm/s)')
axes[2].set_ylabel('Peak height (µm)')
axes[2].set_title('Peak amplitude vs. speed\n(do peaks grow or shrink?)')
axes[2].legend(fontsize=9)

plt.suptitle(f'Peak phase shift with speed  (N_QUANTILES={N_QUANTILES})',
             fontsize=11, fontweight='bold')
plt.tight_layout()
plt.savefig(OUTPUT_DIR / 'height_peak_phase_shift.pdf', bbox_inches='tight')
plt.show()


In [ ]:
### Cell H2d — Stance & swing duration vs. speed (per step cycle)

# Hypothesis: stance shortens with speed; swing duration stays roughly constant.
# This compresses the stance fraction of the cycle and shifts the effective
# liftoff earlier in time — consistent with what we see in the phase plots.

# ── Per-cycle stance / swing durations ───────────────────────────────────────
_cycle_records = []
for (bid, cid), grp in df_valid.dropna(subset=['step_cycle_id']).groupby(['bout_id', 'step_cycle_id'], sort=False):
    if 'T1_left_swing_stance' not in grp.columns:
        continue
    ss        = grp['T1_left_swing_stance'].values
    n_total   = len(ss)
    n_stance  = int(np.sum(ss == 1))
    n_swing   = int(np.sum(ss == 0))
    mean_spd  = grp['step_cycle_mean_speed'].iloc[0]
    if np.isnan(mean_spd) or n_total < 5:
        continue
    _cycle_records.append(dict(
        bout_id         = bid,
        cycle_id        = cid,
        fly_id          = grp['fly_id'].iloc[0],
        mean_speed      = mean_spd,
        stance_ms       = n_stance  / FPS * 1000,
        swing_ms        = n_swing   / FPS * 1000,
        total_ms        = n_total   / FPS * 1000,
        stance_fraction = n_stance  / n_total,
        stride_freq_hz  = FPS / n_total,
    ))

df_cycle = pd.DataFrame(_cycle_records)
print(f"Step cycles analysed: {len(df_cycle):,}  |  "
      f"flies: {df_cycle['fly_id'].nunique()}  |  "
      f"speed range: {df_cycle['mean_speed'].min():.1f}–"
      f"{df_cycle['mean_speed'].max():.1f} mm/s")
print(df_cycle[['stance_ms','swing_ms','total_ms','stance_fraction','stride_freq_hz']].describe().round(2))

# ── Spearman correlations ─────────────────────────────────────────────────────
from scipy.stats import spearmanr as _sr
print("\n── Spearman r  (speed vs. …) ──")
for col, label in [('stance_ms',       'stance duration'),
                   ('swing_ms',         'swing duration'),
                   ('total_ms',         'cycle duration'),
                   ('stance_fraction',  'stance fraction'),
                   ('stride_freq_hz',   'stride frequency')]:
    r, p = _sr(df_cycle['mean_speed'], df_cycle[col], nan_policy='omit')
    print(f"  {label:<20s}  r={r:+.3f}  p={p:.2e}")

# ── Binned means ± SEM (10 equal-quantile bins) ───────────────────────────────
N_SPD_BINS = 10
_spd_bins  = pd.qcut(df_cycle['mean_speed'], q=N_SPD_BINS)
_spd_ctrs  = df_cycle.groupby(_spd_bins, observed=True)['mean_speed'].mean()

def _binned(col):
    g = df_cycle.groupby(_spd_bins, observed=True)[col]
    return g.mean(), g.sem()

# ── Figure ────────────────────────────────────────────────────────────────────
fig, axes = plt.subplots(2, 3, figsize=(18, 10))
axes = axes.flatten()

scatter_kw = dict(s=4, alpha=0.15, rasterized=True)
line_kw    = dict(lw=2.5, marker='o', ms=7, capsize=3, zorder=5)

# Color per fly
_fly_ids   = sorted(df_cycle['fly_id'].unique())
_fly_colors = {fid: plt.cm.tab10(i) for i, fid in enumerate(_fly_ids)}

panels = [
    ('stance_ms',      'Stance duration (ms)',    'Stance shortens with speed?'),
    ('swing_ms',       'Swing duration (ms)',      'Swing stays constant?'),
    ('total_ms',       'Cycle duration (ms)',      'Cycle duration vs. speed'),
    ('stance_fraction','Stance fraction',          'Stance fraction of cycle'),
    ('stride_freq_hz', 'Stride frequency (Hz)',    'Stride frequency vs. speed'),
]

for ax, (col, ylabel, title) in zip(axes[:5], panels):
    # Raw scatter (gray)
    ax.scatter(df_cycle['mean_speed'], df_cycle[col],
               color='gray', **scatter_kw)
    # Binned mean ± SEM
    mn, se = _binned(col)
    ax.errorbar(_spd_ctrs, mn, yerr=se, color='k', **line_kw, label='Mean ± SEM')
    ax.set_xlabel('Step-cycle mean speed (mm/s)')
    ax.set_ylabel(ylabel)
    ax.set_title(title)
    r, p = _sr(df_cycle['mean_speed'], df_cycle[col], nan_policy='omit')
    ax.text(0.97, 0.97, f'r={r:+.2f}\np={p:.1e}',
            transform=ax.transAxes, ha='right', va='top', fontsize=9,
            bbox=dict(fc='white', ec='none', alpha=0.7))
    ax.legend(fontsize=8)

# Panel 6: stance vs swing on same axis to compare directly
ax6 = axes[5]
mn_st, se_st = _binned('stance_ms')
mn_sw, se_sw = _binned('swing_ms')
ax6.errorbar(_spd_ctrs, mn_st, yerr=se_st, color='steelblue',
             label='Stance', **line_kw)
ax6.errorbar(_spd_ctrs, mn_sw, yerr=se_sw, color='tomato',
             label='Swing',  **line_kw)
ax6.set_xlabel('Step-cycle mean speed (mm/s)')
ax6.set_ylabel('Duration (ms)')
ax6.set_title('Stance vs. swing duration (overlay)')
ax6.legend(fontsize=9)

plt.suptitle('T1_left stance & swing duration vs. running speed\n'
             '(each point = one step cycle; line = binned mean ± SEM)',
             fontsize=12, fontweight='bold')
plt.tight_layout()
plt.savefig(OUTPUT_DIR / 'stance_swing_vs_speed.pdf', bbox_inches='tight')
plt.show()


In [ ]:
### Cell H2b — Two-peak significance: per-fly curves + bootstrap CI

z_um     = df_valid['thorax_z_detrended'].values * 1000   # mm → µm
valid_ph = ~np.isnan(df_valid['T1_left_phase'].values)

# ── Per-fly phase averages ────────────────────────────────────────────────────
fly_ids_sorted = sorted(df_valid['fly_id'].unique())
fly_curves = []
for fid in fly_ids_sorted:
    fly_mask = (df_valid['fly_id'].values == fid) & valid_ph
    curve = np.array([
        z_um[fly_mask & (bin_idx == b)].mean() if (fly_mask & (bin_idx == b)).any() else np.nan
        for b in range(N_BINS)
    ])
    fly_curves.append(curve)
fly_curves = np.array(fly_curves)   # (n_flies, N_BINS)

fly_mean = np.nanmean(fly_curves, axis=0)
fly_std  = np.nanstd(fly_curves,  axis=0)

# ── Bootstrap by step cycle ───────────────────────────────────────────────────
_valid_cycles = df_valid.dropna(subset=['step_cycle_id', 'T1_left_phase'])
cycle_groups  = list(_valid_cycles.groupby(['bout_id', 'step_cycle_id']))
n_cycles      = len(cycle_groups)
print(f"Total step cycles for bootstrap: {n_cycles}")

# Precompute per-cycle phase-bin means  (n_cycles × N_BINS)
cycle_mat = np.full((n_cycles, N_BINS), np.nan)
for i, (_, grp) in enumerate(cycle_groups):
    _b = bin_idx[grp.index]   # phase bin per frame (bin_idx is positional, index is 0-based)
    _z = grp['thorax_z_detrended'].values * 1000
    for b in range(N_BINS):
        vals = _z[_b == b]
        if len(vals) > 0:
            cycle_mat[i, b] = np.nanmean(vals)

N_BOOT = 1000
rng    = np.random.default_rng(42)
boot_means = np.full((N_BOOT, N_BINS), np.nan)
for i in range(N_BOOT):
    idx = rng.integers(0, n_cycles, size=n_cycles)
    boot_means[i] = np.nanmean(cycle_mat[idx], axis=0)

boot_lo = np.nanpercentile(boot_means, 2.5,  axis=0)
boot_hi = np.nanpercentile(boot_means, 97.5, axis=0)
overall_mean = np.nanmean(cycle_mat, axis=0)

# ── Plot ──────────────────────────────────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(16, 5))

# Left: per-fly curves + mean ± STD across flies
colors_fly = plt.cm.tab10(np.linspace(0, 0.7, len(fly_ids_sorted)))
for fi, (fid, curve) in enumerate(zip(fly_ids_sorted, fly_curves)):
    axes[0].plot(bin_centers, curve, color=colors_fly[fi], lw=1, alpha=0.6, label=fid)
axes[0].fill_between(bin_centers, fly_mean - fly_std, fly_mean + fly_std,
                     alpha=0.25, color='k', label='±STD across flies')
axes[0].plot(bin_centers, fly_mean, 'k-', lw=2.5, label='Mean across flies')
axes[0].axhline(0, color='k', lw=0.5, ls='--')
axes[0].axvline(-np.pi/2, color='blue', ls=':', lw=1)
if mean_td_phase is not None:
    axes[0].axvline(mean_td_phase, color='orange', ls='--', lw=1)
axes[0].set_xlabel('T1_left phase (rad)'); axes[0].set_ylabel('Detrended thorax Z (µm)')
axes[0].set_title(f'Per-fly phase averages  (N = {len(fly_ids_sorted)} flies)')
axes[0].set_xticks([-np.pi, -np.pi/2, 0, np.pi/2, np.pi])
axes[0].set_xticklabels(['-π', '-π/2', '0', 'π/2', 'π'])
axes[0].legend(fontsize=7, ncol=2)

# Right: bootstrap 95% CI (resampled over step cycles)
axes[1].fill_between(bin_centers, boot_lo, boot_hi, alpha=0.35, label='95% bootstrap CI')
axes[1].plot(bin_centers, overall_mean, lw=2, label='Mean')
axes[1].axhline(0, color='k', lw=0.5, ls='--')
axes[1].axvline(-np.pi/2, color='blue', ls=':', lw=1, label='T1L liftoff (−π/2)')
if mean_td_phase is not None:
    axes[1].axvline(mean_td_phase, color='orange', ls='--', lw=1, label='T1L touchdown')
axes[1].set_xlabel('T1_left phase (rad)'); axes[1].set_ylabel('Detrended thorax Z (µm)')
axes[1].set_title(f'Bootstrap 95% CI  ({N_BOOT} resamples of {n_cycles} cycles)')
axes[1].set_xticks([-np.pi, -np.pi/2, 0, np.pi/2, np.pi])
axes[1].set_xticklabels(['-π', '-π/2', '0', 'π/2', 'π'])
axes[1].legend(fontsize=8)

plt.suptitle('Two-peak effect: per-fly consistency and bootstrap CI',
             fontsize=11, fontweight='bold')
plt.tight_layout()
plt.savefig(OUTPUT_DIR / 'height_phase_twopeak_significance.pdf', bbox_inches='tight')
plt.show()

# ── Fourier decomposition: 2nd harmonic dominance ─────────────────────────────
fft_coefs = np.fft.rfft(fly_mean)
power     = np.abs(fft_coefs)**2
print(f"\nFourier power by harmonic (mean-across-flies curve):")
for k in range(1, 5):
    print(f"  Harmonic {k}: power = {power[k]:.2f}")
h1, h2 = power[1], power[2]
print(f"\n2nd / 1st harmonic power ratio: {h2/h1:.2f}")
print("→ " + ("2nd harmonic dominates: two peaks per stride ✓"
               if h2 > h1 else "1st harmonic dominates: one peak per stride"))

# -- Per-fly grid: individual oscillation +/- within-fly STD ----------------
# For each fly: std of z_um within each phase bin (cycle-to-cycle variability)
fly_stds = []
for fid in fly_ids_sorted:
    fly_mask = (df_valid['fly_id'].values == fid) & valid_ph
    std_curve = np.array([
        z_um[fly_mask & (bin_idx == b)].std() if (fly_mask & (bin_idx == b)).sum() > 1 else np.nan
        for b in range(N_BINS)
    ])
    fly_stds.append(std_curve)
fly_stds = np.array(fly_stds)   # (n_flies, N_BINS)

# Frame counts per fly (for subplot titles)
fly_n = {fid: int((df_valid['fly_id'].values == fid).sum()) for fid in fly_ids_sorted}

n_flies = len(fly_ids_sorted)
ncols   = min(4, n_flies)
nrows   = int(np.ceil(n_flies / ncols))

fig, axes = plt.subplots(nrows, ncols, figsize=(ncols * 4, nrows * 3.2), sharey=True, sharex=True)
axes_flat = np.atleast_1d(np.array(axes)).ravel()

for fi, fid in enumerate(fly_ids_sorted):
    ax         = axes_flat[fi]
    mean_curve = fly_curves[fi]
    std_curve  = fly_stds[fi]
    c          = colors_fly[fi]
    ax.fill_between(bin_centers,
                    mean_curve - std_curve, mean_curve + std_curve,
                    alpha=0.30, color=c)
    ax.plot(bin_centers, mean_curve, color=c, lw=2)
    ax.axhline(0, color='k', lw=0.5, ls='--')
    ax.axvline(-np.pi/2, color='blue', ls=':', lw=0.8, alpha=0.7)
    if mean_td_phase is not None:
        ax.axvline(mean_td_phase, color='orange', ls='--', lw=0.8, alpha=0.7)
    ax.set_title(f'{str(fid).split("/")[-1]}  (n={fly_n[fid]:,})', fontsize=8)
    ax.set_xticks([-np.pi, 0, np.pi])
    ax.set_xticklabels(['-π', '0', 'π'], fontsize=7)
    if fi % ncols == 0:
        ax.set_ylabel('ΔZ (µm)', fontsize=8)
    if fi >= n_flies - ncols:
        ax.set_xlabel('T1L phase (rad)', fontsize=8)

for ax in axes_flat[n_flies:]:
    ax.set_visible(False)

plt.suptitle('Per-fly phase-averaged oscillation  (mean ± within-fly STD)',
             fontsize=11, fontweight='bold')
plt.tight_layout()
plt.savefig(OUTPUT_DIR / 'height_phase_perfly.pdf', bbox_inches='tight')
plt.show()

# -- All bouts ordered shortest to longest: phase-averaged oscillation +/- STD --
bout_lengths_all  = df_valid.groupby('bout_id').size()
all_bouts_ordered = bout_lengths_all.sort_values(ascending=True).index.tolist()

# Per-bout cycle count and mean speed
bout_ncycles    = (df_valid.dropna(subset=['step_cycle_id'])
                   .groupby('bout_id')['step_cycle_id'].nunique())
bout_mean_speed = (df_valid.dropna(subset=['step_cycle_mean_speed'])
                   .groupby('bout_id')['step_cycle_mean_speed'].mean())

all_bout_mean_curves = []
all_bout_std_curves  = []
all_bout_meta        = []   # (bout_id, fly_id, n_frames, n_cycles, mean_speed)

for bid in all_bouts_ordered:
    bmask  = (df_valid['bout_id'].values == bid) & valid_ph
    mean_c = np.array([
        z_um[bmask & (bin_idx == b)].mean() if (bmask & (bin_idx == b)).any() else np.nan
        for b in range(N_BINS)
    ])
    std_c  = np.array([
        z_um[bmask & (bin_idx == b)].std() if (bmask & (bin_idx == b)).sum() > 1 else np.nan
        for b in range(N_BINS)
    ])
    all_bout_mean_curves.append(mean_c)
    all_bout_std_curves.append(std_c)
    fid_b   = df_valid.loc[df_valid['bout_id'] == bid, 'fly_id'].iloc[0]
    n_cyc   = int(bout_ncycles.get(bid, 0))
    spd     = float(bout_mean_speed.get(bid, np.nan))
    all_bout_meta.append((bid, fid_b, int(bout_lengths_all[bid]), n_cyc, spd))

n_bouts_all = len(all_bouts_ordered)
ncols_ab    = 6
nrows_ab    = int(np.ceil(n_bouts_all / ncols_ab))
print(f'Plotting {n_bouts_all} bouts in a {nrows_ab}x{ncols_ab} grid')

fig, axes = plt.subplots(nrows_ab, ncols_ab,
                         figsize=(ncols_ab * 3.5, nrows_ab * 3.2),
                         sharey=False, sharex=True)
axes_flat = np.atleast_1d(np.array(axes)).ravel()

# Color by fly_id so bouts from the same fly share a color
fly_ids_all   = sorted(df_valid['fly_id'].unique())
fly_color_map = {fid: plt.cm.tab10(i / max(len(fly_ids_all) - 1, 1))
                 for i, fid in enumerate(fly_ids_all)}

for bi, (bid, fid_b, n_fr, n_cyc, spd) in enumerate(all_bout_meta):
    ax         = axes_flat[bi]
    mean_curve = all_bout_mean_curves[bi]
    std_curve  = all_bout_std_curves[bi]
    c          = fly_color_map[fid_b]
    ax.fill_between(bin_centers,
                    mean_curve - std_curve, mean_curve + std_curve,
                    alpha=0.30, color=c)
    ax.plot(bin_centers, mean_curve, color=c, lw=1.5)
    ax.axhline(0, color='k', lw=0.4, ls='--')
    ax.axvline(-np.pi/2, color='blue', ls=':', lw=0.7, alpha=0.7)
    if mean_td_phase is not None:
        ax.axvline(mean_td_phase, color='orange', ls='--', lw=0.7, alpha=0.7)
    spd_str = f'{spd:.1f} mm/s' if not np.isnan(spd) else 'n/a'
    ax.set_title(
        f'{bid}  {str(fid_b).split("/")[-1]}\n'
        f'{n_fr:,} fr  |  {n_cyc} cyc  |  {spd_str}',
        fontsize=6)
    ax.set_xticks([-np.pi, 0, np.pi])
    ax.set_xticklabels(['-π', '0', 'π'], fontsize=6)
    ax.tick_params(axis='y', labelsize=6)
    if bi % ncols_ab == 0:
        ax.set_ylabel('ΔZ (µm)', fontsize=7)
    if bi >= n_bouts_all - ncols_ab:
        ax.set_xlabel('T1L phase', fontsize=7)

for ax in axes_flat[n_bouts_all:]:
    ax.set_visible(False)

plt.suptitle(f'All {n_bouts_all} bouts ordered shortest→longest: '
             f'phase-averaged oscillation (mean ± within-bout STD)\n'
             f'color = fly identity',
             fontsize=11, fontweight='bold')
plt.tight_layout()
plt.savefig(OUTPUT_DIR / 'height_phase_allbouts.pdf', bbox_inches='tight')
plt.show()


In [ ]:
### Cell H4 — DC speed-height relationship

bout_stats = df_valid.groupby('bout_id').agg(
    mean_speed=('forward_speed', 'mean'),
    median_speed=('forward_speed', 'median'),
    mean_height=('thorax_z_mm', 'mean'),
    std_height=('thorax_z_mm', 'std'),
    fly_id=('fly_id', 'first'),
).reset_index()
if 'sex' in df_valid.columns:
    bout_stats['sex'] = bout_stats['fly_id'].map(
        df_valid.drop_duplicates('fly_id').set_index('fly_id')['sex'])

r_bout, p_bout = pearsonr(bout_stats['mean_speed'], bout_stats['mean_height'])

r_within = []
for bid, grp in df_valid.groupby('bout_id'):
    if len(grp) < 50: continue
    r, p = spearmanr(grp['forward_speed'], grp['thorax_z_mm'], nan_policy='omit')
    r_within.append({'bout_id': bid, 'r': r, 'p': p,
                     'fly_id': grp['fly_id'].iloc[0],
                     'mean_speed': grp['forward_speed'].mean()})
r_within_df = pd.DataFrame(r_within)

fig, axes = plt.subplots(1, 3, figsize=(18, 5))

colors_fly = {fid: plt.cm.tab10(i) for i, fid in enumerate(bout_stats['fly_id'].unique())}
for fid, grp in bout_stats.groupby('fly_id'):
    axes[0].scatter(grp['mean_speed'], grp['mean_height'],
                    c=[colors_fly[fid]], s=40, alpha=0.7, label=fid)
_xs = np.linspace(bout_stats['mean_speed'].min(), bout_stats['mean_speed'].max(), 100)
_sl, _it = np.polyfit(bout_stats['mean_speed'], bout_stats['mean_height'], 1)
axes[0].plot(_xs, _sl*_xs + _it, 'k--', lw=1.5)
axes[0].set_xlabel('Mean forward speed (mm/s)'); axes[0].set_ylabel('Mean thorax height (mm)')
axes[0].set_title(f'Mean height vs. mean speed\nPearson r={r_bout:.2f}, p={p_bout:.3f}')
axes[0].legend(fontsize=6, ncol=2)

axes[1].hist(r_within_df['r'], bins=30, edgecolor='k')
axes[1].axvline(0, color='k', ls='--')
axes[1].axvline(r_within_df['r'].median(), color='r', ls='-',
                label=f'median r={r_within_df["r"].median():.2f}')
axes[1].set_xlabel('Within-bout Spearman r (speed vs. height)')
axes[1].set_ylabel('N bouts')
axes[1].set_title('Within-bout height-speed correlation')
axes[1].legend()

speed_bins_dc   = pd.qcut(df_valid['forward_speed'], q=10)
height_by_speed = df_valid.groupby(speed_bins_dc, observed=True)['thorax_z_mm'].agg(['mean','sem'])
speed_centers   = df_valid.groupby(speed_bins_dc, observed=True)['forward_speed'].mean()
axes[2].errorbar(speed_centers, height_by_speed['mean'], yerr=height_by_speed['sem'],
                 fmt='o-', capsize=3)
axes[2].set_xlabel('Forward speed bin (mm/s)'); axes[2].set_ylabel('Mean thorax height (mm)')
axes[2].set_title('Height vs. speed (decile bins, all frames)')

plt.suptitle('DC speed-height relationship', fontsize=11, fontweight='bold')
plt.tight_layout()
plt.savefig(OUTPUT_DIR / 'height_speed_dc.pdf', bbox_inches='tight')
plt.show()

print(f"Per-bout Pearson r = {r_bout:.3f} (p = {p_bout:.4f})")
print(f"Within-bout Spearman r: median = {r_within_df['r'].median():.3f}")
sign = 'higher' if r_bout > 0 else 'lower'
print(f"→ Faster flies walk {sign} on average.")

# Store for summary figure
_speed_centers_dc   = speed_centers.copy()
_height_by_speed_dc = height_by_speed.copy()

In [ ]:
# Cell H6 — Amplitude of oscillation vs. speed

from scipy.stats import linregress as _linregress

# Compute speed oscillation inline if Cell S0 hasn't run
if 'forward_speed_osc' not in df_valid.columns:
    df_valid['forward_speed_osc'] = (
        df_valid['forward_speed'] - df_valid['step_cycle_mean_speed'])

cycle_amp = df_valid.groupby(['bout_id', 'step_cycle_id']).agg(
    speed   =('step_cycle_mean_speed', 'first'),
    z_amp   =('thorax_z_detrended',  lambda x: np.sqrt(np.nanmean(x**2)) * 1000),
    spd_amp =('forward_speed_osc',  lambda x: np.sqrt(np.nanmean(x**2))),
    fly_id  =('fly_id', 'first'),
).reset_index().dropna()

fig, axes = plt.subplots(1, 2, figsize=(2 * 38 / 25.4, 32 / 25.4))

for ax, ycol, ylabel, color in [
    (axes[0], 'z_amp',   u'Thorax Z RMS amp. (µm)',  '#f282b4'),
    (axes[1], 'spd_amp', 'Forward speed RMS amp. (mm/s)', '#0011a7'),
]:
    x = cycle_amp['speed'].values
    y = cycle_amp[ycol].values

    # Clip outliers to 1–99th percentile
    xlo, xhi = np.percentile(x, 1), np.percentile(x, 99)
    ylo, yhi = np.percentile(y, 1), np.percentile(y, 99)
    ok = (x >= xlo) & (x <= xhi) & (y >= ylo) & (y <= yhi)
    xc, yc = x[ok], y[ok]

    # Scatter (rasterized at 300 dpi for clean PDF export)
    ax.scatter(xc, yc, s=1, alpha=0.08, color=color, rasterized=True, zorder=1)

    # Binned mean ± STD (8 quantile bins)
    _df  = pd.DataFrame({'speed': xc, 'amp': yc})
    _bins = pd.qcut(_df['speed'], q=8)
    _agg  = _df.groupby(_bins, observed=True)['amp'].agg(['mean', 'std'])
    _spdc = _df.groupby(_bins, observed=True)['speed'].mean()
    ax.errorbar(_spdc, _agg['mean'], yerr=_agg['std'],
                fmt='o', color=color, lw=1.0, ms=3, capsize=2, zorder=5)

    # Linear regression
    sl, ic, r, p, _ = _linregress(xc, yc)
    _xs = np.linspace(xc.min(), xc.max(), 100)
    ax.plot(_xs, sl * _xs + ic, color='k', lw=0.8, ls='--', zorder=6)
    ax.text(0.05, 0.95, f'slope = {sl:.2e}\nr = {r:.2f}',
            transform=ax.transAxes, va='top')

    ax.set_xlabel('Step-cycle mean speed (mm/s)')
    ax.set_ylabel(ylabel)
    sns.despine(ax=ax)

plt.tight_layout()
plt.savefig(OUTPUT_DIR / 'height_oscillation_amplitude.pdf', bbox_inches='tight', dpi=300)
plt.show()


In [ ]:
### Cell H7 — Signal + peak alignment check (longest / shortest 5 bouts)

from scipy.signal import find_peaks as _fp_chk

# Compute speed oscillation inline if Cell S0 hasn't run
if 'forward_speed_osc' not in df_valid.columns:
    df_valid['forward_speed_osc'] = (
        df_valid['forward_speed'] - df_valid['step_cycle_mean_speed'])

# ── Config (mirror Cell P0) ────────────────────────────────────────────────
Z_PROM_CHK        = 0.3    # µm
SPD_PROM_CHK      = 2.0    # mm/s
PEAK_MIN_DIST_CHK = 15     # frames

# ── Per-cycle top-2 peak extraction (exact logic from Cell P0) ─────────────
def _kept_peaks_per_cycle(grp_bout, sig_col, prom):
    """Return bout-level frame indices of the top-2-by-prominence peaks,
    exactly matching Cell P0: per step cycle, skip if mean_spd is NaN,
    mask positions where sig or phase is NaN, require >=6 valid frames,
    keep top-2 by prominence, only if phase at peak is finite."""
    kept = []
    ph_col = 'T1_left_phase'
    valid_cycles = grp_bout.dropna(subset=['step_cycle_id'])
    for cid, cyc in valid_cycles.groupby('step_cycle_id', sort=False):
        if len(cyc) < 10:
            continue
        mean_spd = cyc['step_cycle_mean_speed'].iloc[0]
        if np.isnan(mean_spd):
            continue
        sig   = cyc[sig_col].values
        phase = cyc[ph_col].values
        valid = np.isfinite(sig) & np.isfinite(phase)
        if valid.sum() < 6:
            continue
        s = np.where(valid, sig, 0.0)
        pks, props = _fp_chk(s, prominence=prom, distance=PEAK_MIN_DIST_CHK)
        if len(pks) == 0:
            continue
        order = np.argsort(props['prominences'])[::-1]
        top2  = sorted(pks[order[:2]])
        bout_offset = cyc.index[0] - grp_bout.index[0]
        for p in top2:
            if np.isfinite(phase[p]):                # match P0 final check
                kept.append(bout_offset + p)
    return np.array(kept, dtype=int)

# ── Select bouts ───────────────────────────────────────────────────────────
bout_lengths   = df_valid.groupby('bout_id').size().sort_values()
selected_bouts = bout_lengths.head(5).index.tolist() + bout_lengths.tail(5).index.tolist()
group_labels   = ['shortest'] * 5 + ['longest'] * 5

# ── Helper: shade contiguous True runs ────────────────────────────────────
def _shade_runs(ax, mask, t_ms, color, alpha=0.15):
    in_run, t0 = False, 0
    for i, val in enumerate(mask):
        if val and not in_run:
            t0 = t_ms[i]; in_run = True
        elif not val and in_run:
            ax.axvspan(t0, t_ms[i - 1], color=color, alpha=alpha, lw=0)
            in_run = False
    if in_run:
        ax.axvspan(t0, t_ms[-1], color=color, alpha=alpha, lw=0)

# ── Figure ─────────────────────────────────────────────────────────────────
n_bouts = len(selected_bouts)
fig = plt.figure(figsize=(320 / 25.4, n_bouts * 28 / 25.4))

for row, (bid, glabel) in enumerate(zip(selected_bouts, group_labels)):
    grp   = df_valid[df_valid['bout_id'] == bid].reset_index(drop=True)
    t_ms  = np.arange(len(grp)) / FPS * 1000
    z     = grp['thorax_z_detrended'].values * 1000   # µm
    spd   = grp['forward_speed_osc'].values            # mm/s
    phase = grp['T1_left_phase'].values

    ax_z   = fig.add_subplot(n_bouts, 2, row * 2 + 1)
    ax_spd = fig.add_subplot(n_bouts, 2, row * 2 + 2, sharex=ax_z)

    # ── Signals ───────────────────────────────────────────────────────────
    ax_z.plot(t_ms,   z,   color='#f282b4', lw=0.5)
    ax_spd.plot(t_ms, spd, color='#0011a7', lw=0.5)

    # ── L1 stance shading ─────────────────────────────────────────────────
    stance_mask = np.where(np.isfinite(phase), np.abs(phase) >= np.pi / 2, False)
    _shade_runs(ax_z,   stance_mask, t_ms, color='gray')
    _shade_runs(ax_spd, stance_mask, t_ms, color='gray')

    # ── L1 liftoff lines ──────────────────────────────────────────────────
    for lf in liftoff_from_hilbert(phase):
        for ax in (ax_z, ax_spd):
            ax.axvline(t_ms[lf], color='k', lw=0.4, ls='--', alpha=0.6, zorder=4)

    # ── Kept peaks (top-2 per cycle, matching Cell P0) ────────────────────
    for sig, col, ax, color, prom in [
        (z,   'thorax_z_detrended_um', ax_z,   '#f282b4', Z_PROM_CHK),
        (spd, 'forward_speed_osc',     ax_spd, '#0011a7', SPD_PROM_CHK),
    ]:
        # attach µm-scaled height temporarily for height peaks
        _tmp = grp.copy()
        _tmp['thorax_z_detrended_um'] = z
        pks = _kept_peaks_per_cycle(_tmp, col, prom)
        if len(pks) > 0:
            ax.scatter(t_ms[pks], sig[pks], s=10, color=color,
                       edgecolors='k', linewidths=0.3, zorder=5)

    # ── Labels ────────────────────────────────────────────────────────────
    n_frames = len(grp)
    n_cycles = int(grp.dropna(subset=['step_cycle_id'])['step_cycle_id'].nunique())
    mean_spd = grp['step_cycle_mean_speed'].mean()
    ax_z.set_title(
        f'{glabel.upper()}  |  {bid}  |  {n_frames} frames ' 
        f'({n_frames/FPS*1000:.0f} ms)  |  {n_cycles} cycles  |  ' 
        f'mean speed {mean_spd:.1f} mm/s',
        fontsize=5, loc='left')
    ax_z.set_ylabel('Height (µm)', fontsize=6)
    ax_spd.set_ylabel('Speed osc. (mm/s)', fontsize=6)
    ax_z.tick_params(labelsize=5)
    ax_spd.tick_params(labelsize=5)
    if row == n_bouts - 1:
        ax_z.set_xlabel('Time (ms)', fontsize=6)
        ax_spd.set_xlabel('Time (ms)', fontsize=6)
    else:
        plt.setp(ax_z.get_xticklabels(),   visible=False)
        plt.setp(ax_spd.get_xticklabels(), visible=False)
    sns.despine(ax=ax_z)
    sns.despine(ax=ax_spd)

# ── Legend ─────────────────────────────────────────────────────────────────
from matplotlib.lines import Line2D
from matplotlib.patches import Patch
fig.axes[0].legend(handles=[
    Line2D([0], [0], color='k', lw=0.5, ls='--', label='L1 liftoff'),
    Patch(facecolor='gray', alpha=0.15, label='L1 stance'),
    Line2D([0], [0], marker='o', color='w', markerfacecolor='#f282b4',
           markeredgecolor='k', markersize=4, label='Height peak (top-2/cycle)'),
    Line2D([0], [0], marker='o', color='w', markerfacecolor='#0011a7',
           markeredgecolor='k', markersize=4, label='Speed peak (top-2/cycle)'),
], fontsize=5, loc='upper right', frameon=False)

plt.tight_layout(h_pad=0.4, w_pad=1.0)
_out = OUTPUT_DIR / 'peak_signal_alignment_check.pdf'
plt.savefig(_out, bbox_inches='tight', dpi=150)
plt.show()
print(f'Saved: {_out}')


In [ ]:
### Cell S0 — Phase-averaged forward speed oscillation: per-fly curves + bootstrap CI
# Requires H2 (bin_idx, bin_centers, N_BINS, N_QUANTILES, colors_q, _mean_speed_per_q,
#              df_valid['_sq'], mean_td_phase) and H2b (fly_ids_sorted).

# ── Config ───────────────────────────────────────────────────────────────────
# Choose the error band shown in the main 2-panel figure:
#   'overall_std'  — std across all frames, pooled
#   'fly_std'      — std of the per-fly mean curves
#   'bootstrap_ci' — 95% bootstrap CI resampled over step cycles
DISPLAY_MODE = 'overall_std'   # ← change me

# ── Compute within-cycle speed oscillation ───────────────────────────────────
df_valid['forward_speed_osc'] = (
    df_valid['forward_speed'] - df_valid['step_cycle_mean_speed']
)

spd_osc  = df_valid['forward_speed_osc'].values
valid_ph = ~np.isnan(df_valid['T1_left_phase'].values)
valid_sp = ~np.isnan(spd_osc)
valid    = valid_ph & valid_sp

# ── Per-fly phase averages ────────────────────────────────────────────────────
fly_ids_sorted = sorted(df_valid['fly_id'].unique())
spd_fly_curves = []
for fid in fly_ids_sorted:
    fm = (df_valid['fly_id'].values == fid) & valid
    curve = np.array([
        spd_osc[fm & (bin_idx == b)].mean() if (fm & (bin_idx == b)).any() else np.nan
        for b in range(N_BINS)
    ])
    spd_fly_curves.append(curve)
spd_fly_curves = np.array(spd_fly_curves)
spd_fly_mean   = np.nanmean(spd_fly_curves, axis=0)
spd_fly_std    = np.nanstd(spd_fly_curves,  axis=0)

# ── Overall pooled std ────────────────────────────────────────────────────────
spd_overall_std = np.array([
    spd_osc[valid & (bin_idx == b)].std() if (valid & (bin_idx == b)).any() else np.nan
    for b in range(N_BINS)
])

# ── Bootstrap CI by step cycle ────────────────────────────────────────────────
_valid_spd       = df_valid.dropna(subset=['step_cycle_id', 'T1_left_phase', 'forward_speed_osc'])
spd_cycle_groups = list(_valid_spd.groupby(['bout_id', 'step_cycle_id']))
n_cycles_spd     = len(spd_cycle_groups)
print(f"Step cycles with valid speed oscillation: {n_cycles_spd}")

spd_cycle_mat = np.full((n_cycles_spd, N_BINS), np.nan)
for i, (_, grp) in enumerate(spd_cycle_groups):
    _b = bin_idx[grp.index]
    _s = grp['forward_speed_osc'].values
    for b in range(N_BINS):
        vals = _s[_b == b]
        if len(vals):
            spd_cycle_mat[i, b] = np.nanmean(vals)

N_BOOT = 1000
rng    = np.random.default_rng(42)
spd_boot = np.full((N_BOOT, N_BINS), np.nan)
for i in range(N_BOOT):
    idx = rng.integers(0, n_cycles_spd, size=n_cycles_spd)
    spd_boot[i] = np.nanmean(spd_cycle_mat[idx], axis=0)

spd_boot_lo   = np.nanpercentile(spd_boot, 2.5,  axis=0)
spd_boot_hi   = np.nanpercentile(spd_boot, 97.5, axis=0)
spd_boot_mean = np.nanmean(spd_cycle_mat, axis=0)

# Choose band based on DISPLAY_MODE
if DISPLAY_MODE == 'bootstrap_ci':
    _band_lo, _band_hi = spd_boot_lo, spd_boot_hi
    _band_label = '95% bootstrap CI'
    _center     = spd_boot_mean
elif DISPLAY_MODE == 'fly_std':
    _band_lo, _band_hi = spd_fly_mean - spd_fly_std, spd_fly_mean + spd_fly_std
    _band_label = '±STD across fly means'
    _center     = spd_fly_mean
else:  # 'overall_std'
    _band_lo, _band_hi = spd_fly_mean - spd_overall_std, spd_fly_mean + spd_overall_std
    _band_label = '±overall STD'
    _center     = spd_fly_mean

# Per-quantile averages for S1
_spd_phase_q_stored = {}
for qlabel, grp in df_valid.groupby('_sq', observed=True):
    qm = (df_valid['_sq'] == qlabel).values & valid
    curve = np.array([
        spd_osc[qm & (bin_idx == b)].mean() if (qm & (bin_idx == b)).any() else np.nan
        for b in range(N_BINS)
    ])
    _spd_phase_q_stored[qlabel] = curve

colors_fly = plt.cm.tab10(np.linspace(0, 0.7, len(fly_ids_sorted)))

# ── Paper figure: per-fly curves (gray, smoothed) + black mean ───────────────
_SMOOTH_SIGMA_PLOT = 2   # Gaussian σ over phase bins (soft smoothing)

fig_s0, ax_s0 = plt.subplots(1, 1, figsize=(30 / 25.4, 20 / 25.4))

for curve in spd_fly_curves:
    _smoothed = gaussian_filter1d(np.where(np.isnan(curve), 0, curve), sigma=_SMOOTH_SIGMA_PLOT)
    ax_s0.plot(bin_centers, _smoothed, color='#aaaaaa', lw=0.6, alpha=0.7)

_mean_smoothed = gaussian_filter1d(np.where(np.isnan(spd_fly_mean), 0, spd_fly_mean),
                                    sigma=_SMOOTH_SIGMA_PLOT)
ax_s0.plot(bin_centers, _mean_smoothed, color='k', lw=1.2)
ax_s0.axhline(0, color='k', lw=0.4, ls='--')
ax_s0.axvline(-np.pi/2, color='#4878CF', ls=':', lw=0.6)
if mean_td_phase is not None:
    ax_s0.axvline(mean_td_phase, color='#D65F5F', ls='--', lw=0.6)

ax_s0.set_xticks([-np.pi, -np.pi/2, 0, np.pi/2, np.pi])
ax_s0.set_xticklabels(['-π', '-π/2', '0', 'π/2', 'π'])
ax_s0.set_xlabel('T1L phase (rad)')
ax_s0.set_ylabel('Speed osc. (mm/s)')
import seaborn as _sns; _sns.despine(ax=ax_s0)
plt.tight_layout(pad=0.4)
plt.savefig(OUTPUT_DIR / 'speed_phase_avg_paper.pdf', bbox_inches='tight')
plt.show()

# ── Fourier decomposition ─────────────────────────────────────────────────────
fft_spd   = np.fft.rfft(spd_fly_mean)
power_spd = np.abs(fft_spd)**2
print("\nFourier power by harmonic (mean-across-flies forward speed curve):")
for k in range(1, 5):
    print(f"  Harmonic {k}: power = {power_spd[k]:.2f}")
h1s, h2s = power_spd[1], power_spd[2]
print(f"\n2nd / 1st harmonic power ratio: {h2s/h1s:.2f}")
print("→ " + ("2nd harmonic dominates: two speed peaks per stride ✓"
               if h2s > h1s else "1st harmonic dominates: one peak per stride"))

# ── Per-fly grid ──────────────────────────────────────────────────────────────
spd_fly_stds_g = []
for fid in fly_ids_sorted:
    fm = (df_valid['fly_id'].values == fid) & valid
    spd_fly_stds_g.append(np.array([
        spd_osc[fm & (bin_idx == b)].std() if (fm & (bin_idx == b)).sum() > 1 else np.nan
        for b in range(N_BINS)
    ]))
spd_fly_stds_g = np.array(spd_fly_stds_g)
fly_n = {fid: int((df_valid['fly_id'].values == fid).sum()) for fid in fly_ids_sorted}

n_flies = len(fly_ids_sorted)
ncols_g = min(4, n_flies)
nrows_g = int(np.ceil(n_flies / ncols_g))
fig, axes = plt.subplots(nrows_g, ncols_g, figsize=(ncols_g*4, nrows_g*3.2),
                         sharey=True, sharex=True)
axes_flat = np.atleast_1d(np.array(axes)).ravel()
for fi, fid in enumerate(fly_ids_sorted):
    ax = axes_flat[fi]
    mc, sc = spd_fly_curves[fi], spd_fly_stds_g[fi]
    c = colors_fly[fi]
    ax.fill_between(bin_centers, mc - sc, mc + sc, alpha=0.30, color=c)
    ax.plot(bin_centers, mc, color=c, lw=2)
    ax.axhline(0, color='k', lw=0.5, ls='--')
    ax.axvline(-np.pi/2, color='royalblue',  ls=':', lw=0.8, alpha=0.7)
    if mean_td_phase is not None:
        ax.axvline(mean_td_phase, color='darkorange', ls='--', lw=0.8, alpha=0.7)
    ax.set_title(f'{str(fid).split("/")[-1]}  (n={fly_n[fid]:,})', fontsize=8)
    ax.set_xticks([-np.pi, 0, np.pi]); ax.set_xticklabels(['-π','0','π'], fontsize=7)
    if fi % ncols_g == 0: ax.set_ylabel('Speed osc. (mm/s)', fontsize=8)
    if fi >= n_flies - ncols_g: ax.set_xlabel('T1L phase (rad)', fontsize=8)
for ax in axes_flat[n_flies:]: ax.set_visible(False)
plt.suptitle('Per-fly phase-averaged forward speed oscillation  (mean ± within-fly STD)',
             fontsize=11, fontweight='bold')
plt.tight_layout()
plt.savefig(OUTPUT_DIR / 'speed_phase_perfly.pdf', bbox_inches='tight')
plt.show()

# ── Per-bout grid ─────────────────────────────────────────────────────────────
bout_lengths_all  = df_valid.groupby('bout_id').size()
all_bouts_ordered = bout_lengths_all.sort_values().index.tolist()
bout_ncycles_spd  = (_valid_spd.groupby('bout_id')['step_cycle_id'].nunique())
bout_mean_speed   = (df_valid.dropna(subset=['step_cycle_mean_speed'])
                     .groupby('bout_id')['step_cycle_mean_speed'].mean())
fly_color_map     = {fid: plt.cm.tab10(i / max(len(fly_ids_sorted)-1, 1))
                     for i, fid in enumerate(fly_ids_sorted)}

n_bouts  = len(all_bouts_ordered)
ncols_b  = 6
nrows_b  = int(np.ceil(n_bouts / ncols_b))
print(f'Plotting {n_bouts} bouts in a {nrows_b}x{ncols_b} grid')

fig, axes = plt.subplots(nrows_b, ncols_b, figsize=(ncols_b*3.5, nrows_b*3.2),
                         sharey=False, sharex=True)
axes_flat = np.atleast_1d(np.array(axes)).ravel()

for bi, bid in enumerate(all_bouts_ordered):
    ax      = axes_flat[bi]
    bmask   = (df_valid['bout_id'].values == bid) & valid
    mean_c  = np.array([
        spd_osc[bmask & (bin_idx == b)].mean() if (bmask & (bin_idx == b)).any() else np.nan
        for b in range(N_BINS)
    ])
    std_c   = np.array([
        spd_osc[bmask & (bin_idx == b)].std() if (bmask & (bin_idx == b)).sum() > 1 else np.nan
        for b in range(N_BINS)
    ])
    fid_b   = df_valid.loc[df_valid['bout_id'] == bid, 'fly_id'].iloc[0]
    c       = fly_color_map[fid_b]
    n_cyc   = int(bout_ncycles_spd.get(bid, 0))
    spd_val = float(bout_mean_speed.get(bid, np.nan))

    ax.fill_between(bin_centers, mean_c - std_c, mean_c + std_c, alpha=0.30, color=c)
    ax.plot(bin_centers, mean_c, color=c, lw=1.5)
    ax.axhline(0, color='k', lw=0.4, ls='--')
    ax.axvline(-np.pi/2, color='royalblue',  ls=':', lw=0.7, alpha=0.7)
    if mean_td_phase is not None:
        ax.axvline(mean_td_phase, color='darkorange', ls='--', lw=0.7, alpha=0.7)
    spd_str = f'{spd_val:.1f} mm/s' if not np.isnan(spd_val) else 'n/a'
    ax.set_title(f'{bid}  {str(fid_b).split("/")[-1]}\n'
                 f'{int(bout_lengths_all[bid]):,} fr  |  {n_cyc} cyc  |  {spd_str}',
                 fontsize=6)
    ax.set_xticks([-np.pi, 0, np.pi]); ax.set_xticklabels(['-π','0','π'], fontsize=6)
    ax.tick_params(axis='y', labelsize=6)
    if bi % ncols_b == 0: ax.set_ylabel('Speed osc. (mm/s)', fontsize=7)
    if bi >= n_bouts - ncols_b: ax.set_xlabel('T1L phase', fontsize=7)

for ax in axes_flat[n_bouts:]: ax.set_visible(False)
plt.suptitle(f'All {n_bouts} bouts ordered shortest→longest: '
             f'phase-averaged forward speed oscillation (mean ± within-bout STD)\n'
             f'color = fly identity',
             fontsize=11, fontweight='bold')
plt.tight_layout()
plt.savefig(OUTPUT_DIR / 'speed_phase_allbouts.pdf', bbox_inches='tight')
plt.show()


In [ ]:
### Cell S1 — Forward speed peak phases vs speed
# Requires S0 (_spd_phase_q_stored, _mean_speed_per_q, colors_q, bin_centers, N_BINS)

from scipy.signal import find_peaks as _fp2

speed_peak_records = []
for qi, (qlabel, qmean) in enumerate(_spd_phase_q_stored.items()):
    if np.all(np.isnan(qmean)):
        continue
    pks, _ = _fp2(np.nan_to_num(qmean), prominence=1.0, distance=N_BINS // 5)
    mean_spd = _mean_speed_per_q.get(qlabel, np.nan)
    for pk in pks:
        speed_peak_records.append(dict(
            quantile   = qlabel,
            mean_speed = mean_spd,
            peak_bin   = pk,
            peak_phase = bin_centers[pk],
            peak_height= float(qmean[pk]),
        ))

speed_peak_df = pd.DataFrame(speed_peak_records)
print(speed_peak_df.to_string(index=False))

marker_styles = ['o', 's', '^', 'D', 'v', 'P']
fig, axes = plt.subplots(1, 2, figsize=(16, 5))

# Left: quantile curves + peak markers
for qi, (qlabel, qmean) in enumerate(_spd_phase_q_stored.items()):
    color  = colors_q[qi]
    marker = marker_styles[qi % len(marker_styles)]
    axes[0].plot(bin_centers, qmean, color=color, lw=2, label=qlabel)
    sub = speed_peak_df[speed_peak_df['quantile'] == qlabel]
    for _, row in sub.iterrows():
        axes[0].plot(row['peak_phase'], row['peak_height'],
                     marker=marker, ms=10, color=color,
                     markeredgecolor='k', markeredgewidth=0.8, zorder=5)
axes[0].axhline(0, color='k', lw=0.5, ls='--')
axes[0].axvline(-np.pi/2, color='royalblue',  ls=':', lw=1, alpha=0.6,
                label='T1L liftoff (−π/2)')
axes[0].set_xlabel('T1_left phase (rad)')
axes[0].set_ylabel('Forward speed oscillation (mm/s)')
axes[0].set_title('Phase-averaged speed per quantile + peak markers')
axes[0].set_xticks([-np.pi, -np.pi/2, 0, np.pi/2, np.pi])
axes[0].set_xticklabels(['-π', '-π/2', '0', 'π/2', 'π'])
axes[0].legend(fontsize=8)

# Right: peak phase vs mean speed
if not speed_peak_df.empty:
    for qi, (qlabel, grp) in enumerate(speed_peak_df.groupby('quantile',
                                                               observed=True, sort=False)):
        axes[1].scatter(grp['mean_speed'], grp['peak_phase'],
                        color=colors_q[qi], s=100, label=qlabel, zorder=3)
    axes[1].axhline(0,         color='k',      ls='--', lw=0.8)
    axes[1].axhline( np.pi/2,  color='darkorange', ls=':', lw=1, label='+π/2 touchdown')
    axes[1].axhline(-np.pi/2,  color='royalblue',  ls=':', lw=1, label='−π/2 liftoff')
    axes[1].set_xlabel('Mean running speed (mm/s)')
    axes[1].set_ylabel('Speed-peak phase (rad)')
    axes[1].set_title('Phase of forward speed peaks vs speed')
    axes[1].set_yticks([-np.pi, -np.pi/2, 0, np.pi/2, np.pi])
    axes[1].set_yticklabels(['-π', '-π/2', '0', 'π/2', 'π'])
    axes[1].legend(fontsize=8)

plt.suptitle('Forward speed peak phases vs running speed', fontsize=11, fontweight='bold')
plt.tight_layout()
plt.savefig(OUTPUT_DIR / 'speed_peak_phase_vs_speed.pdf', bbox_inches='tight')
plt.show()


In [ ]:
### Cell S2 — Phase comparison: scutellum height peaks vs forward speed peaks
# Requires H2 (_z_phase_q_stored, bin_centers, N_BINS, _mean_speed_per_q, colors_q)
# and S1 (speed_peak_df).
# H2c (peak_df) is used if already run; otherwise height peaks are recomputed here.

from scipy.stats import spearmanr as _sr
from scipy.signal import find_peaks as _fp_h

# ── Ensure peak_df (height peaks) is available ───────────────────────────────
if 'peak_df' not in globals():
    print("peak_df not found — recomputing height peaks from _z_phase_q_stored")
    _hpk_records = []
    for qi, (qlabel, qmean) in enumerate(_z_phase_q_stored.items()):
        if np.all(np.isnan(qmean)): continue
        _prom = max(3.0, 0.15 * float(np.nanmax(qmean) - np.nanmin(qmean)))
        pks, _ = _fp_h(np.nan_to_num(qmean), prominence=_prom, distance=N_BINS // 5)
        mean_spd = _mean_speed_per_q.get(qlabel, np.nan)
        for pk in pks:
            _hpk_records.append(dict(quantile=qlabel, mean_speed=mean_spd,
                                     peak_bin=int(pk), peak_phase=float(bin_centers[pk]),
                                     peak_height=float(qmean[pk])))
    peak_df = pd.DataFrame(_hpk_records)
    print(f"  → {len(peak_df)} height peaks found")
else:
    print(f"Using peak_df from H2c  ({len(peak_df)} height peaks)")

def _circ_diff(a, b):
    return float((a - b + np.pi) % (2 * np.pi) - np.pi)

# ── Match height ↔ speed peaks per quantile ──────────────────────────────────
comparison_records = []
all_quantiles = sorted(set(peak_df['quantile'].tolist() +
                           speed_peak_df['quantile'].tolist()))

for qlabel in all_quantiles:
    hpks = peak_df[peak_df['quantile'] == qlabel]['peak_phase'].values
    spks = speed_peak_df[speed_peak_df['quantile'] == qlabel]['peak_phase'].values
    mean_spd = _mean_speed_per_q.get(qlabel, np.nan)
    if len(hpks) == 0 or len(spks) == 0:
        continue
    for hp in hpks:
        diffs   = [_circ_diff(hp, sp) for sp in spks]
        best_sp = spks[np.argmin(np.abs(diffs))]
        comparison_records.append(dict(
            quantile   = qlabel,
            mean_speed = mean_spd,
            phi_height = float(hp),
            phi_speed  = float(best_sp),
            delta_phi  = _circ_diff(hp, best_sp),
        ))

comp_df = pd.DataFrame(comparison_records)
print("\n── Phase comparison summary ─────────────────────────────────────────")
print(comp_df.to_string(index=False))

_valid_comp = comp_df.dropna(subset=['mean_speed', 'delta_phi'])
if len(_valid_comp) >= 4:
    r_s, p_s = _sr(_valid_comp['mean_speed'], _valid_comp['delta_phi'])
    print(f"\nSpearman r = {r_s:.3f},  p = {p_s:.4f}")
    print("→ " + ("speed-dependent phase shift → consistent with running" if p_s < 0.05
                  else "no significant phase shift → walking-like at all speeds"))
else:
    r_s, p_s = np.nan, np.nan
    print("\nToo few points for Spearman test.")

# ── Figure ────────────────────────────────────────────────────────────────────
fig = plt.figure(figsize=(18, 5))
ax0 = fig.add_subplot(1, 3, 1)
ax1 = fig.add_subplot(1, 3, 2)
ax2 = fig.add_subplot(1, 3, 3, projection='polar')

# Left: both peak phases vs speed
for qi, (qlabel, grp) in enumerate(comp_df.groupby('quantile', observed=True, sort=False)):
    c = colors_q[qi]
    ax0.scatter(grp['mean_speed'], grp['phi_height'],
                color=c, s=100, marker='o', zorder=3, label=f'{qlabel} height')
    ax0.scatter(grp['mean_speed'], grp['phi_speed'],
                color=c, s=100, marker='^', zorder=3, label=f'{qlabel} speed')
    for _, row in grp.iterrows():
        ax0.plot([row['mean_speed']]*2, [row['phi_height'], row['phi_speed']],
                 color=c, lw=0.8, alpha=0.5)
ax0.axhline(0, color='k', ls='--', lw=0.8)
ax0.axhline( np.pi/2, color='darkorange', ls=':', lw=1, label='+π/2 touchdown')
ax0.axhline(-np.pi/2, color='royalblue',  ls=':', lw=1, label='−π/2 liftoff')
ax0.set_xlabel('Mean speed (mm/s)'); ax0.set_ylabel('Peak phase (rad)')
ax0.set_title('Height peaks (○) vs speed peaks (△)\nvs running speed')
ax0.set_yticks([-np.pi,-np.pi/2,0,np.pi/2,np.pi])
ax0.set_yticklabels(['-π','-π/2','0','π/2','π'])
ax0.legend(fontsize=7, ncol=2)

# Middle: Δφ vs speed
for qi, (qlabel, grp) in enumerate(comp_df.groupby('quantile', observed=True, sort=False)):
    ax1.scatter(grp['mean_speed'], grp['delta_phi'],
                color=colors_q[qi], s=100, label=qlabel, zorder=3)
if len(_valid_comp) >= 4:
    from scipy.stats import linregress as _lr
    _xs = np.linspace(_valid_comp['mean_speed'].min(), _valid_comp['mean_speed'].max(), 100)
    _sl, _ic, *_ = _lr(_valid_comp['mean_speed'], _valid_comp['delta_phi'])
    ax1.plot(_xs, _sl*_xs + _ic, 'k--', lw=1.5,
             label=f'fit  r={r_s:.2f}  p={p_s:.3f}')
ax1.axhline(0, color='k', lw=1.2, alpha=0.4, label='Δφ=0 (in phase)')
ax1.set_xlabel('Mean speed (mm/s)'); ax1.set_ylabel('Δφ = φ_height − φ_speed (rad)')
ax1.set_title('Phase difference vs speed\n(0 = in phase; drift = running signature)')
ax1.set_yticks([-np.pi,-np.pi/2,0,np.pi/2,np.pi])
ax1.set_yticklabels(['-π','-π/2','0','π/2','π'])
ax1.legend(fontsize=8)

# Right: polar — where each signal sits on the phase circle
for qi, (qlabel, grp) in enumerate(comp_df.groupby('quantile', observed=True, sort=False)):
    c = colors_q[qi]
    for _, row in grp.iterrows():
        ax2.scatter(row['phi_height'], 1.0, color=c, s=80, marker='o', zorder=3)
        ax2.scatter(row['phi_speed'],  0.6, color=c, s=80, marker='^', zorder=3, alpha=0.8)
        ax2.annotate('', xy=(row['phi_speed'], 0.6), xytext=(row['phi_height'], 1.0),
                     arrowprops=dict(arrowstyle='->', color=c, lw=1.0, alpha=0.6))
ax2.set_theta_zero_location('N'); ax2.set_theta_direction(-1); ax2.set_rticks([])
ax2.set_xticks(np.linspace(0, 2*np.pi, 5)[:-1])
ax2.set_xticklabels(['-π/0', '+π/2 (TD)', '+π', '-π/2 (LO)'], fontsize=8)
ax2.set_title('Phase positions on unit circle\n'
              '○=height peak  △=speed peak\n(outer→inner = slow→fast)', fontsize=8, pad=15)

_stat_str = f'Spearman r={r_s:.2f}, p={p_s:.3f}' if not np.isnan(r_s) else ''
plt.suptitle(f'Height vs speed peak phase  |  {_stat_str}\n'
             f'constant Δφ → walking  |  Δφ grows with speed → running',
             fontsize=11, fontweight='bold')
plt.tight_layout()
plt.savefig(OUTPUT_DIR / 'speed_height_phase_comparison.pdf', bbox_inches='tight')
plt.show()


In [ ]:
### Cell P0 — Per-cycle dual-peak phase extraction → df_pt
# Requires: H2 (N_QUANTILES, _q_labels), H2d (step_cycle_id), S0 (forward_speed_osc)

from scipy.signal import find_peaks

# ── Config ────────────────────────────────────────────────────────────────
N_Q_POLAR  = 3     # ← change: number of speed quantiles for polar plots
Z_PROM     = 0.3   # µm  — height peak prominence threshold
SPD_PROM   = 2.0   # mm/s — speed peak prominence threshold
PEAK_MIN_DIST = 15 # frames — min separation between peaks within a cycle

_q_labels_polar = (
    ['slow', 'medium', 'fast'] if N_Q_POLAR == 3
    else ['slow', 'fast'] if N_Q_POLAR == 2
    else [f'Q{i+1}' for i in range(N_Q_POLAR)]
)

# ── Extract per-cycle top-2 peaks ─────────────────────────────────────────
pt_records = []
for (bid, cid), grp in (
    df_valid.dropna(subset=['step_cycle_id']).groupby(['bout_id', 'step_cycle_id'], sort=False)
):
    if len(grp) < 10:
        continue
    phase    = grp['T1_left_phase'].values
    fly_id   = grp['fly_id'].iloc[0]
    mean_spd = grp['step_cycle_mean_speed'].iloc[0]
    if np.isnan(mean_spd):
        continue

    signals = [
        ('height', grp['thorax_z_detrended'].values * 1000, Z_PROM),
        ('speed',  grp['forward_speed_osc'].values,         SPD_PROM),
    ]
    for sig_name, sig, prom in signals:
        valid = np.isfinite(sig) & np.isfinite(phase)
        if valid.sum() < 6:
            continue
        s = np.where(valid, sig, 0.0)
        pks, props = find_peaks(s, prominence=prom, distance=PEAK_MIN_DIST)
        if len(pks) == 0:
            continue
        # Top-2 by prominence, then sort by position (early/late)
        order   = np.argsort(props['prominences'])[::-1]
        top2    = sorted(pks[order[:2]])  # sorted by frame index = early/late
        for rank, pk in enumerate(top2):
            if np.isfinite(phase[pk]):
                pt_records.append(dict(
                    signal=sig_name, peak_rank=rank,
                    phase=float(phase[pk]),
                    fly_id=fly_id, mean_speed=mean_spd,
                ))

df_pt = pd.DataFrame(pt_records)
df_pt['speed_q'] = pd.qcut(
    df_pt['mean_speed'], q=N_Q_POLAR, labels=_q_labels_polar
)
quantile_order_polar = _q_labels_polar
colors_q_polar = plt.cm.viridis(np.linspace(0.2, 0.9, N_Q_POLAR))

print(f'df_pt: {len(df_pt)} peaks')
print(df_pt.groupby(['signal', 'peak_rank']).size().rename('n').to_string())

In [ ]:
### Cell P1 — Polar rose plots: one per speed quantile, height vs. speed overlaid
# Each polar: blue = height peaks, red = speed peaks (both early + late pooled into one rose).
# Arrows = circular mean per (signal, peak_rank).  ○ = early, △ = late.

from scipy.stats import circmean

N_PBINS    = 24
bin_edges  = np.linspace(-np.pi, np.pi, N_PBINS + 1)
bin_width  = 2 * np.pi / N_PBINS
theta_ctrs = 0.5 * (bin_edges[:-1] + bin_edges[1:])

HEIGHT_COLOR = '#f282b4';  HEIGHT_ALPHA = 1.0
SPEED_COLOR  = '#0011a7';  SPEED_ALPHA  = round(0x7c / 255, 3)  # 0.486

POLAR_TRIPOD_PHASE = False   # True → mark mean tripod LO/TD; False → T1_left labels
TRIPOD_LEGS = ['T1_left', 'T2_right', 'T3_left']

if POLAR_TRIPOD_PHASE:
    _lo_phases_trip, _td_phases_trip = [], []
    for _bid, _grp in df_valid.groupby('bout_id'):
        _t1l = _grp['T1_left_phase'].values
        for _leg in TRIPOD_LEGS:
            _pc = f'{_leg}_phase'
            if _pc not in _grp.columns:
                continue
            _lp = _grp[_pc].values
            for _i in liftoff_from_hilbert(_lp):
                if _i < len(_t1l) and np.isfinite(_t1l[_i]):
                    _lo_phases_trip.append(_t1l[_i])
            for _i in landing_from_hilbert(_lp):
                if _i < len(_t1l) and np.isfinite(_t1l[_i]):
                    _td_phases_trip.append(_t1l[_i])
    _trip_lo = circmean(_lo_phases_trip, low=-np.pi, high=np.pi)
    _trip_td = circmean(_td_phases_trip, low=-np.pi, high=np.pi)

fig, axes = plt.subplots(1, N_Q_POLAR, figsize=(120 * N_Q_POLAR / 25.4, 60 / 25.4),
                          subplot_kw={'projection': 'polar'})
if N_Q_POLAR == 1:
    axes = [axes]

# Scale markers/arrows with figure height
_sc = fig.get_figheight() / (30 / 25.4)  # 1.0 at 30 mm reference
DOT_S     = max(2, 8  * _sc)
ARROW_LW  = max(0.3, 0.75 * _sc)

# ── Shared rlim: compute global max across all quantiles + signals ───────────
_global_rose_max = 0.0
for _ql in quantile_order_polar:
    _sub_tmp = df_pt[df_pt['speed_q'] == _ql]
    for _sig in ['height', 'speed']:
        _ph_tmp = _sub_tmp[_sub_tmp['signal'] == _sig]['phase'].values
        if len(_ph_tmp) == 0: continue
        _c_tmp, _ = np.histogram(_ph_tmp, bins=bin_edges)
        _global_rose_max = max(_global_rose_max, (_c_tmp / _c_tmp.sum()).max())
_rlim_max = _global_rose_max * 1.15

for qi, qlabel in enumerate(quantile_order_polar):
    ax  = axes[qi]
    sub = df_pt[df_pt['speed_q'] == qlabel]

    # ── Compute roses (normalised to fraction of peaks per bin) ──────────────
    rose_max = 0.0
    fracs = {}
    for sig_name in ['height', 'speed']:
        phases = sub[sub['signal'] == sig_name]['phase'].values
        if len(phases) == 0:
            fracs[sig_name] = np.zeros(N_PBINS)
            continue
        counts, _ = np.histogram(phases, bins=bin_edges)
        f = counts / counts.sum()
        fracs[sig_name] = f
        rose_max = max(rose_max, f.max())

    for sig_name, color, label in [
        ('height', HEIGHT_COLOR, 'Height'),
        ('speed',  SPEED_COLOR,  'Speed'),
    ]:
        ax.bar(theta_ctrs, fracs[sig_name], width=bin_width * 0.9,
               color=color, alpha=HEIGHT_ALPHA if sig_name == 'height' else SPEED_ALPHA,
               label=label if qi == 0 else '')


    ax.set_rlim(0, _rlim_max)
    _r_ticks = [t for t in [0.10, 0.5] if t <= _rlim_max]
    ax.set_yticks(_r_ticks)
    ax.set_yticklabels([f'{t:.2f}' for t in _r_ticks])
    ax.tick_params(axis='y', labelsize=plt.rcParams['font.size'])
    ax.set_rlabel_position(67.5)  # place label in top-right gap
    _r_dot = _rlim_max * 0.88   # slightly inside edge so markers don't overflow

    # ── Labels in white space (below circle) ─────────────────────────────
    n_h = (sub['signal'] == 'height').sum()
    n_s = (sub['signal'] == 'speed').sum()
    _spd_lo = sub['mean_speed'].min()
    _spd_hi = sub['mean_speed'].max()
    ax.text(0.5, -0.12, f'{qlabel}  {_spd_lo:.0f}–{_spd_hi:.0f} mm/s  h={n_h} s={n_s}',
            transform=ax.transAxes, ha='center', va='top',
            fontsize=plt.rcParams['font.size'])

    # ── Per-fly circular means + overall arrows ───────────────────────────────
    # rank 0 = early (○, solid arrow), rank 1 = late (△, dashed arrow)
    for sig_name, color in [('height', HEIGHT_COLOR), ('speed', SPEED_COLOR)]:
        sig_sub = sub[sub['signal'] == sig_name]
        for rank, marker, r_frac, ls in [
            (0, 'o', 1.00, '-'),
            (1, 'o', 1.00, '-'),
        ]:
            rank_sub = sig_sub[sig_sub['peak_rank'] == rank]
            for fid in fly_ids_sorted:
                fly_ph = rank_sub[rank_sub['fly_id'] == fid]['phase'].values
                if len(fly_ph) < 2:
                    continue
                cm = circmean(fly_ph, low=-np.pi, high=np.pi)
                ax.scatter(cm, _r_dot * r_frac, s=DOT_S, color=color,
                           marker=marker, zorder=5, alpha=0.85,
                           edgecolors='none')
            all_ph = rank_sub['phase'].values
            if len(all_ph) >= 3:
                cm = circmean(all_ph, low=-np.pi, high=np.pi)
                ax.annotate('', xy=(cm, _r_dot * r_frac * 0.90), xytext=(0, 0),
                            arrowprops=dict(arrowstyle='->', color=color,
                                            lw=ARROW_LW, linestyle=ls))

    # ── Axis formatting ───────────────────────────────────────────────────────
    ax.set_theta_zero_location('N')
    ax.set_theta_direction(-1)
    ax.set_xticks(np.linspace(0, 2 * np.pi, 5)[:-1])
    if POLAR_TRIPOD_PHASE:
        ax.set_xticklabels(['0\n(swing)', 'π/2', 'π\n(stance)', '-π/2'])
        ax.axvline(_trip_lo, color='k', lw=0.75, linestyle='--', alpha=0.7, zorder=6)
        ax.axvline(_trip_td, color='k', lw=0.75, linestyle=':', alpha=0.7, zorder=6)
    else:
        ax.set_xticklabels(['0\n(swing)', 'π/2\n(TD)', 'π\n(stance)', '-π/2\n(LO)'])

plt.tight_layout()
plt.savefig(OUTPUT_DIR / 'peak_phase_polar_paper.pdf', bbox_inches='tight')
plt.show()

In [ ]:
### Cell A0 — Per-cycle amplitude dataframe → df_amp

from scipy.signal import find_peaks

# ── Config ────────────────────────────────────────────────────────────────
AMP_METHOD = 'range'  # 'range'      : max - min over whole cycle
                            # 'inter_peak' : mean(pk1-trough, pk2-trough)
                            #                where trough is between the two peaks

Z_PROM_A        = 0.3    # µm  (only used for inter_peak)
SPD_PROM_A      = 2.0    # mm/s
PEAK_MIN_DIST_A = 15     # frames
A_MIN_SPEED     = 0.5    # mm/s

# ── Amplitude helpers ─────────────────────────────────────────────────────
def _amp_range(s):
    """max - min over the full signal array."""
    return float(s.max() - s.min())

def _amp_inter_peak(s, prom, min_dist):
    """Mean of (pk1-trough, pk2-trough) using the trough between the two
    highest-prominence peaks. Returns None if fewer than 2 peaks found."""
    pks, props = find_peaks(s, prominence=prom, distance=min_dist)
    if len(pks) < 2:
        return None
    order    = np.argsort(props['prominences'])[::-1]
    pk0, pk1 = sorted(pks[order[:2]])
    trough   = s[pk0:pk1 + 1].min()
    return float((s[pk0] - trough + s[pk1] - trough) / 2)

# ── Main loop ─────────────────────────────────────────────────────────────
_amp_records = []
for (bid, cid), grp in (
    df_valid.dropna(subset=['step_cycle_id'])
    .groupby(['bout_id', 'step_cycle_id'], sort=False)
):
    if len(grp) < 10:
        continue
    mean_spd = grp['step_cycle_mean_speed'].iloc[0]
    if np.isnan(mean_spd):
        continue

    amps = {}
    for col, prom, key, scale in [
        ('thorax_z_detrended', Z_PROM_A,   'A_vert',  1000.0),  # mm → µm for inter_peak
        ('forward_speed',      SPD_PROM_A, 'A_speed',    1.0),
    ]:
        raw   = grp[col].values * scale
        valid = np.isfinite(raw)
        if valid.sum() < 6:
            break
        s = np.where(valid, raw, 0.0)

        if AMP_METHOD == 'range':
            A = _amp_range(s)
        else:
            A = _amp_inter_peak(s, prom, PEAK_MIN_DIST_A)
            if A is None:
                break   # fewer than 2 peaks — skip cycle
        amps[key] = A
    else:
        if amps.get('A_speed', 0) < A_MIN_SPEED:
            continue
        _amp_records.append(dict(
            bout_id    = bid,
            cycle_id   = cid,
            fly_id     = grp['fly_id'].iloc[0],
            mean_speed = mean_spd,
            A_vert     = amps['A_vert']  / 1000,
            A_speed    = amps['A_speed'],
            R          = amps['A_vert'] / 1000 / amps['A_speed'],
        ))

df_amp = pd.DataFrame(_amp_records)
if df_amp.empty:
    raise RuntimeError('df_amp is empty — check that H1 and H2d have run.')
df_amp = df_amp[np.isfinite(df_amp['R'])].reset_index(drop=True)

print(f'AMP_METHOD = {AMP_METHOD!r}   n={len(df_amp)} step cycles')
print(f'  A_vert  (mm):   median={df_amp["A_vert"].median():.4f}')
print(f'  A_speed (mm/s): median={df_amp["A_speed"].median():.3f}')
print(f'  R       (s):    median={df_amp["R"].median():.6f}')

In [ ]:
### Cell A1 — Amplitude ratio: R vs speed, normalized scaling, regression
# Per-fly binned analysis: one line per fly → honest N, visible consistency.

from scipy.stats import pearsonr, spearmanr, linregress, ttest_1samp

HEIGHT_COLOR = '#f282b4'
SPEED_COLOR  = '#0011a7'
N_BINS_AMP   = 8

# ── Speed grid ────────────────────────────────────────────────────────────────
_spd_edges = np.quantile(df_amp['mean_speed'], np.linspace(0, 1, N_BINS_AMP + 1))
_spd_edges[0] -= 0.01; _spd_edges[-1] += 0.01
_spd_centers = (_spd_edges[:-1] + _spd_edges[1:]) / 2

# ── Per-fly binned means ──────────────────────────────────────────────────────
fly_ids_amp = sorted(df_amp['fly_id'].unique())
_n_flies    = len(fly_ids_amp)

fly_R  = np.full((_n_flies, N_BINS_AMP), np.nan)
fly_Av = np.full((_n_flies, N_BINS_AMP), np.nan)
fly_As = np.full((_n_flies, N_BINS_AMP), np.nan)

for fi, fid in enumerate(fly_ids_amp):
    fd = df_amp[df_amp['fly_id'] == fid]
    bi = np.digitize(fd['mean_speed'].values, _spd_edges[1:-1])
    for b in range(N_BINS_AMP):
        sel = fd[bi == b]
        if len(sel) >= 1:
            fly_R[fi, b]  = sel['R'].mean()
            fly_Av[fi, b] = sel['A_vert'].mean()
            fly_As[fi, b] = sel['A_speed'].mean()

def _mean_sem(mat):
    m = np.nanmean(mat, axis=0)
    s = np.nanstd(mat, axis=0) / np.sqrt(np.sum(~np.isnan(mat), axis=0).clip(1))
    return m, s

R_mean,  R_sem  = _mean_sem(fly_R)
Av_mean, Av_sem = _mean_sem(fly_Av)
As_mean, As_sem = _mean_sem(fly_As)

def norm01(x):
    rng = np.nanmax(x) - np.nanmin(x)
    return (x - np.nanmin(x)) / rng if rng > 0 else x * 0

fly_Av_norm = np.array([norm01(fly_Av[fi]) for fi in range(_n_flies)])
fly_As_norm = np.array([norm01(fly_As[fi]) for fi in range(_n_flies)])
Avn_mean, Avn_sem = _mean_sem(fly_Av_norm)
Asn_mean, Asn_sem = _mean_sem(fly_As_norm)

# ── Stats ─────────────────────────────────────────────────────────────────────
fly_R_grand   = np.nanmean(fly_R, axis=1)
fly_spd_grand = np.array([df_amp[df_amp['fly_id']==fid]['mean_speed'].mean()
                           for fid in fly_ids_amp])
r_p, p_p = pearsonr(fly_spd_grand, fly_R_grand)

fly_R_slopes, fly_R_slope_se, _fly_slope_fi = [], [], []
for fi in range(_n_flies):
    mask = ~np.isnan(fly_R[fi])
    if mask.sum() >= 2:
        sl, _, _, _, se = linregress(_spd_centers[mask], fly_R[fi][mask])
        fly_R_slopes.append(sl); fly_R_slope_se.append(se); _fly_slope_fi.append(fi)
fly_R_slopes   = np.array(fly_R_slopes)
fly_R_slope_se = np.array(fly_R_slope_se)
t_R, p_R = ttest_1samp(fly_R_slopes, 0) if len(fly_R_slopes) >= 2 else (np.nan, np.nan)

_valid_mean = ~np.isnan(R_mean)
_sl_mean, _ic_mean, _, _, _se_mean = linregress(
    _spd_centers[_valid_mean], R_mean[_valid_mean])
_xs = np.linspace(_spd_centers[_valid_mean].min(), _spd_centers[_valid_mean].max(), 100)
_ci = 1.96 * _se_mean * np.sqrt(1 + 1/(_valid_mean.sum()) +
      (_xs - _spd_centers[_valid_mean].mean())**2 /
      np.sum((_spd_centers[_valid_mean] - _spd_centers[_valid_mean].mean())**2))

# ── Publication rcParams ──────────────────────────────────────────────────────
import matplotlib as mpl
mpl.rcParams.update({'font.family': 'Arial', 'font.size': 6,
                     'axes.linewidth': 0.5, 'xtick.major.width': 0.5,
                     'ytick.major.width': 0.5, 'xtick.major.size': 2,
                     'ytick.major.size': 2, 'pdf.fonttype': 42})

fig, axes = plt.subplots(1, 3, figsize=(3 * 38 / 25.4, 32 / 25.4))
_sc      = fig.get_figheight() / 5
_fly_lw  = max(0.2, 0.6 * _sc)
_mean_lw = max(0.4, 1.5 * _sc)
_mean_ms = max(1.5, 4   * _sc)
_reg_lw  = max(0.3, 1.2 * _sc)
_cap     = max(1,   2   * _sc)
_big_s   = max(5,   50  * _sc)
_diam_ms = max(2,   6   * _sc)
_sig = lambda p: '***' if p<0.001 else '**' if p<0.01 else '*' if p<0.05 else 'ns'

# ── Panel 0: R vs speed — per-fly lines + mean ± SEM + regression ────────────
_fly_cmap = plt.cm.cool(np.linspace(0.15, 0.85, max(_n_flies, 1)))
ax = axes[0]
for fi in range(_n_flies):
    mask = ~np.isnan(fly_R[fi])
    if mask.sum() >= 2:
        ax.plot(_spd_centers[mask], fly_R[fi][mask],
                color=_fly_cmap[fi], lw=_fly_lw, alpha=0.5, zorder=2)
ax.errorbar(_spd_centers[_valid_mean], R_mean[_valid_mean], yerr=R_sem[_valid_mean],
            fmt='o', color=HEIGHT_COLOR, ms=_mean_ms, lw=_mean_lw, capsize=_cap,
            elinewidth=_mean_lw, zorder=4)
ax.plot(_xs, _sl_mean * _xs + _ic_mean, color='k', lw=_reg_lw, zorder=6)
ax.fill_between(_xs, _sl_mean * _xs + _ic_mean - _ci,
                      _sl_mean * _xs + _ic_mean + _ci,
                alpha=0.15, color='k')
_r_lo, _r_hi = np.percentile(df_amp['R'], [1, 99])
ax.set_ylim(_r_lo, _r_hi)
ax.set_xlabel('Mean step-cycle speed (mm/s)')
ax.set_ylabel('R = A$_{vert}$ / A$_{speed}$  (s)')
ax.text(0.97, 0.97, f'slope {_sig(p_R)}', transform=ax.transAxes,
        ha='right', va='top', fontsize=5)

# ── Panel 1: Per-fly R regression lines ─────────────────────────────────────
ax = axes[1]
_spd_global = np.array([_spd_centers[_valid_mean].min(),
                         _spd_centers[_valid_mean].max()])
for _fi_c, fid in enumerate(fly_ids_amp):
    fd = df_amp[df_amp['fly_id'] == fid]
    if len(fd) >= 2:
        _sl_f, _ic_f, _, _, _ = linregress(fd['mean_speed'].values,
                                            fd['R'].values)
        _xf = np.array([fd['mean_speed'].min(), fd['mean_speed'].max()])
        ax.plot(_xf, _sl_f * _xf + _ic_f,
                color=_fly_cmap[_fi_c], lw=_fly_lw * 1.3, alpha=0.5, zorder=2)
# Mean regression (same line as panel 0)
ax.plot(_spd_global, _sl_mean * _spd_global + _ic_mean,
        color='k', lw=_reg_lw * 1.2, zorder=5)
ax.fill_between(_xs, _sl_mean * _xs + _ic_mean - _ci,
                      _sl_mean * _xs + _ic_mean + _ci,
                alpha=0.12, color='k')
_r_lo1, _r_hi1 = np.percentile(df_amp['R'], [1, 99])
ax.set_ylim(_r_lo1, _r_hi1)
ax.set_xlabel('Mean step-cycle speed (mm/s)')
ax.set_ylabel('R = A$_{vert}$ / A$_{speed}$  (s)')

# ── Panel 2: Per-fly R slopes — dot plot ─────────────────────────────────────
fly_sl_v, fly_sl_s = [], []
for fi in range(_n_flies):
    mask = ~np.isnan(fly_Av[fi]) & ~np.isnan(fly_As[fi])
    if mask.sum() >= 2:
        sl_v, *_ = linregress(_spd_centers[mask], fly_Av[fi][mask])
        sl_s, *_ = linregress(_spd_centers[mask], fly_As[fi][mask])
        fly_sl_v.append(sl_v); fly_sl_s.append(sl_s)
fly_sl_v = np.array(fly_sl_v)
fly_sl_s = np.array(fly_sl_s)
_ratio = np.median(fly_sl_v / fly_sl_s) if len(fly_sl_s) else 0

ax = axes[2]
_rng = np.random.default_rng(1)
_jx  = _rng.uniform(-0.25, 0.25, len(fly_R_slopes))
for _i, (_x, _y, _ye) in enumerate(zip(_jx, fly_R_slopes, fly_R_slope_se)):
    _c = _fly_cmap[_fly_slope_fi[_i]]
    ax.scatter([_x], [_y], color=_c, s=_big_s, edgecolors='none', alpha=0.5, zorder=3)
    ax.errorbar(_x, _y, yerr=_ye, fmt='none',
                ecolor=_c, lw=_fly_lw, zorder=2)
ax.errorbar(0, fly_R_slopes.mean(), yerr=fly_R_slopes.std(),
            fmt='D', color='k', ms=_diam_ms, capsize=_cap, lw=_mean_lw, zorder=5)
ax.axhline(0, color='k', lw=_fly_lw, ls='--', alpha=0.5)
ax.set_xlim(-1, 1)
ax.set_xticks([])
ax.ticklabel_format(axis='y', style='sci', scilimits=(0, 0))
ax.set_ylabel('dR/d(speed)  [s\u00b7(mm/s)\u207b\xb2]')

_spd_range   = _spd_centers[_valid_mean].max() - _spd_centers[_valid_mean].min()
_median_R    = float(df_amp["R"].median())
_frac_change = fly_R_slopes.mean() * _spd_range / _median_R * 100

print("=" * 60)
print("AMPLITUDE RATIO  R = A_vert / A_speed")
print(f"  N flies:   {len(fly_R_slopes)}")
print(f"  Median R:  {_median_R:.5f} s")
print(f"  slope t-test: t={t_R:.3f}, p={p_R:.4g}  {_sig(p_R)}")
print(f"  fractional dR: {_frac_change:+.1f}% over {_spd_range:.1f} mm/s")
print(f"  slope ratio (median dAv/dAs): {_ratio:.6f} s")
print("=" * 60)

plt.tight_layout()
plt.savefig(OUTPUT_DIR / 'amplitude_ratio.pdf', bbox_inches='tight')
plt.show()


In [ ]:
### Cell A1 schematic — visual aid explaining panel 3 variables

import matplotlib as mpl
import matplotlib.gridspec as mgridspec
from scipy.stats import linregress as _linreg

mpl.rcParams.update({'font.family': 'Arial', 'font.size': 6,
                     'axes.linewidth': 0.5, 'xtick.major.width': 0.5,
                     'ytick.major.width': 0.5, 'xtick.major.size': 2,
                     'ytick.major.size': 2, 'pdf.fonttype': 42})

fig = plt.figure(figsize=(80/25.4, 36/25.4))
gs  = mgridspec.GridSpec(2, 2, figure=fig,
                         width_ratios=[1.1, 1], height_ratios=[1, 1],
                         hspace=0.10, wspace=0.55,
                         left=0.13, right=0.97, top=0.88, bottom=0.16)
ax_h  = fig.add_subplot(gs[0, 0])
ax_s  = fig.add_subplot(gs[1, 0], sharex=ax_h)
ax_ex = fig.add_subplot(gs[:, 1])

# ── Synthetic stride cycle: clean concept illustration ──────────────────────
_t = np.linspace(0, 1, 300)
_h = np.sin(2 * np.pi * _t)          # height (A_vert = 1 a.u.)
_v = np.sin(2 * np.pi * _t + 0.25)  # fwd speed (A_speed = 1 a.u., slight lag)

for _ax, _sig, _col, _lbl, _ylabel in [
    (ax_h, _h, HEIGHT_COLOR, 'A$_{vert}$',  'Scutellum\nheight'),
    (ax_s, _v, SPEED_COLOR,  'A$_{speed}$', 'Fwd speed'),
]:
    _ax.plot(_t, _sig, color=_col, lw=1.0)
    _ax.axhline(0, color='k', lw=0.3, ls='--', alpha=0.35)
    # peak-to-trough double arrow
    _ax.annotate('', xy=(0.27, 0.93), xytext=(0.27, -0.93),
                 arrowprops=dict(arrowstyle='<->', color='k', lw=0.7,
                                 mutation_scale=5))
    _ax.text(0.30, 0.0, _lbl, va='center', ha='left', fontsize=5.5)
    _ax.set_yticks([])
    _ax.set_ylim(-1.75, 1.75)
    _ax.set_ylabel(_ylabel, fontsize=5, labelpad=2)
    for sp in ['top', 'right']: _ax.spines[sp].set_visible(False)

ax_h.tick_params(bottom=False, labelbottom=False)
ax_h.spines['bottom'].set_visible(False)
ax_s.set_xticks([0, 0.5, 1])
ax_s.set_xticklabels(['0', '½', '1'])
ax_s.set_xlabel('Stride phase', fontsize=5, labelpad=2)

# R definition — placed between the two left panels
_bx = dict(fc='white', ec='#bbbbbb', lw=0.4, boxstyle='round,pad=0.3')
ax_s.text(0.96, 1.04, 'R = A$_{vert}$ / A$_{speed}$  [s]',
          transform=ax_s.transAxes, ha='right', va='bottom',
          fontsize=5.5, bbox=_bx)

# ── Right: R vs speed for the most-sampled fly ──────────────────────────────
_fly_counts = {fid: int((df_amp['fly_id'] == fid).sum()) for fid in fly_ids_amp}
_ex_fid     = max(_fly_counts, key=_fly_counts.get)
_ex_fi      = list(fly_ids_amp).index(_ex_fid)
_mask_ex    = ~np.isnan(fly_R[_ex_fi])

ax_ex.scatter(_spd_centers[_mask_ex], fly_R[_ex_fi][_mask_ex],
              color=HEIGHT_COLOR, s=12, edgecolors='none', zorder=3)

if _mask_ex.sum() >= 2:
    _sl_ex, _ic_ex, _, _, _ = _linreg(
        _spd_centers[_mask_ex], fly_R[_ex_fi][_mask_ex])
    _xex = np.array([_spd_centers[_mask_ex].min(), _spd_centers[_mask_ex].max()])
    ax_ex.plot(_xex, _sl_ex * _xex + _ic_ex, color='k', lw=0.9, zorder=4)
    # annotate the slope and link it to panel 3
    _xm = float(_spd_centers[_mask_ex].mean())
    _ym = float(_sl_ex * _xm + _ic_ex)
    ax_ex.annotate(f'slope = {_sl_ex:.2e}\n→ one dot\n   in panel 3',
                   xy=(_xm, _ym),
                   xytext=(0.55, 0.22), textcoords='axes fraction',
                   arrowprops=dict(arrowstyle='->', color='k', lw=0.55,
                                   connectionstyle='arc3,rad=0.2'),
                   fontsize=4.5, ha='left', va='top',
                   bbox=dict(fc='white', ec='none', alpha=0.85))

ax_ex.set_xlabel('Step-cycle speed (mm/s)', fontsize=5, labelpad=2)
ax_ex.set_ylabel('R (s)', fontsize=5)
ax_ex.set_title('R vs speed — one fly', fontsize=5.5, pad=2)
for sp in ['top', 'right']: ax_ex.spines[sp].set_visible(False)

fig.suptitle('Each stride cycle: measure A$_{vert}$ (height oscillation) '
             'and A$_{speed}$ (speed oscillation), compute R = A$_{vert}$/A$_{speed}$',
             fontsize=4.5, y=0.995)

plt.savefig(OUTPUT_DIR / 'amplitude_ratio_schematic.pdf', bbox_inches='tight')
plt.show()


In [ ]:
### Cell J0 -- Keypoint noise floor: static-frame analysis on raw JARVIS predictions
#
# Loads Scutellum Z directly from data3D.csv for frames where the fly is stationary,
# then characterises three noise metrics:
#   raw_std       -- total spread (includes slow drift within the interval)
#   detrended_std -- spread after removing a linear ramp (removes slow settling)
#   diff_std      -- std of frame-to-frame differences; unaffected by drift;
#                    best estimate of true instantaneous jitter
#                    (for white noise: diff_std = sqrt(2) * sigma_noise)

# -- Configuration -----------------------------------------------------------
STATIC_PREDICTION_DIRS = [
    Path('/home/user/src/JARVIS-HybridNet/projects/fly50_V5/predictions/predictions3D/Predictions_3D_20260203-103416'),
    Path('/home/user/src/JARVIS-HybridNet/projects/fly50_V5/predictions/predictions3D/Predictions_3D_20260209-094844'),
    Path('/home/user/src/JARVIS-HybridNet/projects/fly50_V5/predictions/predictions3D/Predictions_3D_20260202-171900'),
    Path('/home/user/src/JARVIS-HybridNet/projects/fly50_V5/predictions/predictions3D/BPN-S1/Predictions_3D_20260218-145022/')
]
FPS_RAW         = 800
SCUTELLUM_Z_COL = 'Scutellum.2'   # after pd.read_csv(skiprows=[1]): x=Scutellum, y=.1, z=.2, conf=.3

# -- Helpers ------------------------------------------------------------------
def parse_stationary_frames(folder):
    """Return [(start_frame, end_frame), ...] from stationary_frames[/frame].
    Handles both 'start end' and 'id start end' (tab- or space-separated) formats.
    """
    for fname in ('stationary_frames', 'stationary_frame'):
        p = folder / fname
        if p.exists():
            intervals = []
            for line in p.read_text().splitlines():
                parts = line.split()
                if len(parts) >= 2:
                    intervals.append((int(parts[-2]), int(parts[-1])))
            return intervals
    return []


def compute_noise_metrics(z):
    """Noise characterisation for a 1-D array of Z values (mm).

    Returns dict with raw_std, detrended_std, diff_std, mean_z, n_frames,
    or None if the segment is too short.
    """
    z = np.asarray(z, dtype=float)
    z = z[~np.isnan(z)]
    if len(z) < 4:
        return None
    t     = np.arange(len(z))
    z_lin = z - np.polyval(np.polyfit(t, z, 1), t)
    return dict(
        mean_z        = float(np.mean(z)),
        raw_std       = float(np.std(z)),
        detrended_std = float(np.std(z_lin)),
        diff_std      = float(np.std(np.diff(z))),
        n_frames      = len(z),
    )

# -- Main loop: load each CSV, compute metrics per static interval ------------
noise_records = []
z_seg_cache   = []   # stores raw z segment (mm) for each accepted interval

for folder in STATIC_PREDICTION_DIRS:
    if not folder.exists():
        print(f'  Skipping (not found): {folder.name}')
        continue
    intervals = parse_stationary_frames(folder)
    if not intervals:
        print(f'  No stationary_frames file in {folder.name}')
        continue
    print(f'Loading {folder.name}  ({len(intervals)} static intervals)...', end=' ', flush=True)
    df_z  = pd.read_csv(folder / 'data3D.csv', skiprows=[1],
                        usecols=[SCUTELLUM_Z_COL], low_memory=False)
    z_all = df_z[SCUTELLUM_Z_COL].values.astype(float) / SCALE  # JARVIS raw -> mm (same scale as IK thorax_z_mm)
    print(f'{len(z_all):,} frames total')
    for i, (s, e) in enumerate(intervals):
        m = compute_noise_metrics(z_all[s : e + 1])
        if m:
            m.update(folder=folder.name, interval_idx=i, start=s, end=e)
            noise_records.append(m)
            z_seg_cache.append(z_all[s : e + 1])

noise_df = pd.DataFrame(noise_records)

# -- Noise floor summary ------------------------------------------------------
# Short intervals (< 500 frames) are most reliable: the fly is truly static
# and has not started settling. Longer intervals may include slow positional drift.
noise_floor_short   = noise_df.loc[noise_df['n_frames'] < 500, 'diff_std'].median()
noise_floor_all     = noise_df['diff_std'].median()
noise_floor_detrend = noise_df['detrended_std'].median()

print('\n-- Per-folder noise summary ------------------------------------------')
print(f"  {'Folder':<44}  {'N':>3}  {'diff_std (mean+/-std)':>22}  {'detrend_std (mean+/-std)':>26}")
for fname, grp in noise_df.groupby('folder'):
    print(f'  {fname:<44}  {len(grp):>3}  '
          f'{grp["diff_std"].mean():.4f} +/- {grp["diff_std"].std():.4f}    '
          f'{grp["detrended_std"].mean():.4f} +/- {grp["detrended_std"].std():.4f}')

print(f'\nNoise floor  diff_std  (all intervals):         {noise_floor_all:.4f} mm')
print(f'Noise floor  diff_std  (short <500 frames):     {noise_floor_short:.4f} mm  <- most conservative')
print(f'Noise floor  detrend_std (all):                 {noise_floor_detrend:.4f} mm')
print(f'\nInstantaneous sigma ~ diff_std / sqrt(2) ~ {noise_floor_short / np.sqrt(2):.4f} mm '
      '(assumes frame-to-frame noise is white/uncorrelated)')

# -- Figure -------------------------------------------------------------------
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
folder_labels = noise_df['folder'].unique()
folder_colors = {f: plt.cm.tab10(i) for i, f in enumerate(folder_labels)}

# Left: noise metrics per interval
for fname, grp in noise_df.groupby('folder'):
    c  = folder_colors[fname]
    xi = np.array([noise_df.index.get_loc(i) for i in grp.index])
    axes[0].scatter(xi - 0.15, grp['detrended_std'], c=[c], s=70, marker='o', zorder=3,
                    label=f'{fname[-18:]} detrend')
    axes[0].scatter(xi + 0.15, grp['diff_std'],      c=[c], s=70, marker='^', alpha=0.7, zorder=3,
                    label=f'{fname[-18:]} diff')
axes[0].axhline(noise_floor_all,   color='k',    ls='--', lw=1.5,
                label=f'diff_std (all)   = {noise_floor_all:.3f} mm')
axes[0].axhline(noise_floor_short, color='navy', ls=':',  lw=1.5,
                label=f'diff_std (short) = {noise_floor_short:.3f} mm')
axes[0].set_xlabel('Static interval index (pooled across folders)')
axes[0].set_ylabel('Noise std (mm)')
axes[0].set_title('Scutellum Z noise per static interval\n(circle = detrended_std,  triangle = diff_std)')
axes[0].legend(fontsize=7)

# Right: interval length vs. noise metrics
# Shows that raw_std grows with interval length (drift) while diff_std stays flat
for fname, grp in noise_df.groupby('folder'):
    c = folder_colors[fname]
    axes[1].scatter(grp['n_frames'], grp['raw_std'],  c=[c], s=50, marker='o', alpha=0.8,
                    label=f'{fname[-18:]} raw_std')
    axes[1].scatter(grp['n_frames'], grp['diff_std'], c=[c], s=50, marker='^', alpha=0.8)
axes[1].axhline(noise_floor_all,   color='k',    ls='--', lw=1.5,
                label=f'diff_std floor = {noise_floor_all:.3f} mm')
axes[1].axhline(noise_floor_short, color='navy', ls=':',  lw=1.5)
axes[1].set_xlabel('Interval length (frames)')
axes[1].set_ylabel('Noise metric (mm)')
axes[1].set_title('raw_std grows with interval length (slow drift)\ndiff_std stays flat -> true instantaneous jitter')
axes[1].legend(fontsize=7)

plt.suptitle('JARVIS Scutellum Z jitter in static frames', fontsize=11, fontweight='bold')
plt.tight_layout()
plt.savefig(OUTPUT_DIR / 'jitter_noise_floor.pdf', bbox_inches='tight')
plt.show()

# -- Z vs. frame: raw trace for every static interval -----------------------
n_int  = len(noise_df)
ncols_t = min(5, n_int)
nrows_t = int(np.ceil(n_int / ncols_t))

fig, axes = plt.subplots(nrows_t, ncols_t,
                         figsize=(ncols_t * 4, nrows_t * 2.8),
                         sharey=False)
axes_flat = np.atleast_1d(np.array(axes)).ravel()

for ri, (row, z_seg) in enumerate(zip(noise_df.itertuples(), z_seg_cache)):
    ax     = axes_flat[ri]
    frames = np.arange(len(z_seg))
    c      = folder_colors.get(row.folder, 'steelblue')

    ax.plot(frames, z_seg, lw=0.8, color=c, alpha=0.85)

    # Mean and raw std band
    ax.axhline(row.mean_z,              color='k',   lw=1.2, ls='-')
    ax.axhline(row.mean_z + row.raw_std, color='r',  lw=0.8, ls='--', alpha=0.7)
    ax.axhline(row.mean_z - row.raw_std, color='r',  lw=0.8, ls='--', alpha=0.7)

    # Linear trend
    t_seg  = np.arange(len(z_seg))
    poly   = np.polyfit(t_seg, z_seg, 1)
    ax.plot(t_seg, np.polyval(poly, t_seg), color='orange', lw=1.2, alpha=0.9)

    ax.set_title(
        f'{row.folder[-20:]}\n'
        f'[{row.start}:{row.end}]  n={row.n_frames}\n'
        f'raw_std={row.raw_std:.4f}  diff_std={row.diff_std:.4f}',
        fontsize=6)
    ax.set_xlabel('Frame (within interval)', fontsize=6)
    ax.set_ylabel('Z (mm)', fontsize=6)
    ax.tick_params(labelsize=6)

for ax in axes_flat[n_int:]:
    ax.set_visible(False)

plt.suptitle(
    'Raw Scutellum Z during static intervals\n'
    'black = mean  |  red dashed = mean ± raw_std  |  orange = linear trend',
    fontsize=10, fontweight='bold')
plt.tight_layout()
plt.savefig(OUTPUT_DIR / 'jitter_z_traces.pdf', bbox_inches='tight')
plt.show()


In [ ]:
### Diagnostic — running cycles of a chosen bout: kinematics overview

import matplotlib.font_manager as _fmd
import matplotlib as _mpld
_fmd.fontManager.addfont("/usr/share/fonts/truetype/msttcorefonts/Arial.ttf")
_mpld.rcParams.update({'font.family': 'Arial', 'font.size': 6,
                        'axes.labelsize': 6, 'axes.titlesize': 6,
                        'xtick.labelsize': 6, 'ytick.labelsize': 6,
                        'pdf.fonttype': 42})

# ── Toggles ───────────────────────────────────────────────────────────────────
BOUT_ID          = 'bout_249'   # ← change me
CYCLE_START      = 7            # ← first cycle index (0-based within bout)
N_CYCLES         = 2           # ← how many cycles (None = all)
SMOOTH_SIGMA_Z   = 2            # ← Gaussian σ for scutellum height (frames; 0 = off)
SMOOTH_SIGMA_FWD = 3            # ← Gaussian σ for forward velocity  (frames; 0 = off)
SMOOTH_SIGMA_TIP = 3            # ← Gaussian σ for leg tip speeds    (frames)

# ── Data window ───────────────────────────────────────────────────────────────
b = df_valid[df_valid['bout_id'] == BOUT_ID].copy()
if b.empty:
    raise ValueError(f"{BOUT_ID} not found.")

all_cyc_ids = sorted(b.dropna(subset=['step_cycle_id'])['step_cycle_id'].unique())
_end        = (CYCLE_START + N_CYCLES) if N_CYCLES is not None else len(all_cyc_ids)
cyc_ids     = all_cyc_ids[CYCLE_START:_end]
b_win       = b[b['step_cycle_id'].isin(cyc_ids)].copy()

frames = b_win['frame'].values
stance = (b_win['T1_left_swing_stance'].values.astype(float)
          if 'T1_left_swing_stance' in b_win.columns
          else np.full(len(frames), np.nan))
lo_frames = frames[np.where(np.diff(stance, prepend=stance[0]) == -1)[0]]
td_frames = frames[np.where(np.diff(stance, prepend=stance[0]) ==  1)[0]]

# ── Signal helpers ────────────────────────────────────────────────────────────
from scipy.ndimage import gaussian_filter1d as _gf1d
from scipy.signal  import hilbert as _hilbert

def _smooth(sig, sigma):
    if sigma and sigma > 0:
        fin = np.isfinite(sig)
        if (~fin).any() and fin.sum() > 1:
            sig = sig.copy()
            sig[~fin] = np.interp(np.flatnonzero(~fin), np.flatnonzero(fin), sig[fin])
        return _gf1d(sig, sigma)
    return sig

def _tip_speed(leg_prefix):
    cols = [f'{leg_prefix}_tip_{ax}_world' for ax in ('x', 'y', 'z')]
    xyz  = np.stack([b_win[c].values.astype(float) for c in cols], axis=1)
    for d in range(3):
        col = xyz[:, d]; nans = np.isnan(col)
        if nans.any() and (~nans).sum() > 1:
            col[nans] = np.interp(np.flatnonzero(nans), np.flatnonzero(~nans), col[~nans])
    spd = np.linalg.norm(np.gradient(xyz, axis=0) * FPS_IK, axis=1)
    return _smooth(spd, SMOOTH_SIGMA_TIP)

def _hilbert_phase(sig):
    s = sig - np.nanmean(sig)
    t = np.arange(len(s))
    s = s - np.polyval(np.polyfit(t, np.where(np.isnan(s), 0., s), 1), t)
    s[np.isnan(s)] = 0.
    return np.angle(_hilbert(s))

FPS_IK = 900

z_det   = _smooth(b_win['thorax_z_detrended'].values * 1000, SMOOTH_SIGMA_Z)
fwd_vel = _smooth((b_win['forward_speed'] - b_win['step_cycle_mean_speed']).values,
                  SMOOTH_SIGMA_FWD)
t1l_spd = _tip_speed('T1_left')
t1r_spd = _tip_speed('T1_right')
ph_z    = _hilbert_phase(z_det)
ph_fwd  = _hilbert_phase(fwd_vel)
ph_t1l  = _hilbert_phase(t1l_spd)
ph_t1r  = _hilbert_phase(t1r_spd)

print(f"{BOUT_ID}  cycles {cyc_ids[0]}–{cyc_ids[-1]}  {len(frames)} frames @ {FPS_IK} Hz")

# ── Shared plot helpers ───────────────────────────────────────────────────────
def _shade(ax):
    prev_f, prev_st = frames[0], stance[0]
    for k in range(1, len(frames)):
        if stance[k] != prev_st or k == len(frames) - 1:
            ax.axvspan(prev_f, frames[k], alpha=0.25,
                       color='#cce5cc' if prev_st == 1 else '#fff3cd', lw=0)
            prev_f, prev_st = frames[k], stance[k]

def _events(ax):
    for lf in lo_frames:
        ax.axvline(lf, color='#1f77b4', ls='--', lw=0.5, alpha=0.6)
    for td in td_frames:
        ax.axvline(td, color='#ff7f0e', ls='--', lw=0.5, alpha=0.6)

def _cycles(ax):
    for cid in cyc_ids:
        cf = b_win[b_win['step_cycle_id'] == cid]['frame'].values
        if len(cf):
            ax.axvline(cf[0], color='k', ls=':', lw=0.5, alpha=0.35)

# ── Figure ────────────────────────────────────────────────────────────────────
_W = 150 / 25.4
fig, axes = plt.subplots(4, 1, figsize=(_W, 4 * 22 / 25.4), sharex=True)
ax_z, ax_fwd, ax_leg, ax_ph = axes

# Panel 1 — Detrended thorax height
_shade(ax_z); _events(ax_z); _cycles(ax_z)
ax_z.plot(frames, z_det, color=HEIGHT_COLOR, lw=0.8)
ax_z.set_ylabel('thorax Z detrended (µm)')
sns.despine(ax=ax_z)

# Panel 2 — Forward velocity oscillation
_shade(ax_fwd); _events(ax_fwd); _cycles(ax_fwd)
ax_fwd.plot(frames, fwd_vel, color=SPEED_COLOR, lw=0.8)
ax_fwd.set_ylabel('fwd speed osc. (mm/s)')
sns.despine(ax=ax_fwd)

# Panel 3 — T1 leg tip speeds
_shade(ax_leg); _events(ax_leg); _cycles(ax_leg)
ax_leg.plot(frames, t1l_spd, color='#2ca02c', lw=0.8, label='T1L')
ax_leg.plot(frames, t1r_spd, color='#ff7f0e', lw=0.8, alpha=0.85, label='T1R')
ax_leg.set_ylabel('tip speed (mm/s)')
ax_leg.set_ylim(bottom=0)
ax_leg.legend(fontsize=5, frameon=False, loc='upper right')
sns.despine(ax=ax_leg)

# Panel 4 — Hilbert phase
_shade(ax_ph); _events(ax_ph); _cycles(ax_ph)
for _ph, _clr, _lbl in [
    (ph_z,   HEIGHT_COLOR, 'thorax Z'),
    (ph_fwd, SPEED_COLOR,  'fwd speed'),
    (ph_t1l, '#2ca02c',    'T1L tip'),
    (ph_t1r, '#ff7f0e',    'T1R tip'),
]:
    ax_ph.plot(frames, _ph, color=_clr, lw=0.7, alpha=0.85, label=_lbl)
ax_ph.axhline(-np.pi/2, color='#1f77b4', ls=':', lw=0.5, alpha=0.5)
ax_ph.axhline( np.pi/2, color='#ff7f0e', ls=':', lw=0.5, alpha=0.5)
ax_ph.set_yticks([-np.pi, -np.pi/2, 0, np.pi/2, np.pi])
ax_ph.set_yticklabels(['-π', '-π/2', '0', 'π/2', 'π'])
ax_ph.set_ylabel('Hilbert phase')
ax_ph.set_xlabel(f'frame  ({FPS_IK} Hz)')
ax_ph.legend(fontsize=5, frameon=False, loc='upper right')
sns.despine(ax=ax_ph)

fig.suptitle(f'{BOUT_ID}  ·  cycles {cyc_ids[0]}–{cyc_ids[-1]}  ({len(cyc_ids)} cycles)',
             fontsize=6, y=1.002)
plt.tight_layout(pad=0.3, h_pad=0.4)
plt.savefig(OUTPUT_DIR / 'traces.pdf', bbox_inches='tight', dpi=300)
plt.show()
